# Taller 7 · Pipeline para detectar y corregir errores médicos (MEDIQA-CORR 2024)

**Procesamiento de Lenguaje Natural · Maestría · Grupo 6 · David Lasso, Nestor Fadiño y Camilo García**

Este notebook implementa el pipeline que propusimos en el reporte (`README.md` §4 e `informe/taller7_grupo6.pdf`).
Cada sección es una **caja** del diagrama y define **las mismas funciones** de la tabla del reporte, con los
mismos hiperparámetros. Al final llenamos la fila «Nuestro pipeline» de la tabla de referencia (§5).

| Caja del plan | Funciones | Sección |
|---|---|---|
| 1 · Carga y limpieza | `cargar_corpus`, `parsear_oraciones`, `unir_fragmentos`, `mapear_ids_oficiales` | 1 |
| 2 · Partición por caso | `identificar_casos`, `dividir_por_caso` | 2 |
| 8 · Métricas (se definen antes: cada caja se evalúa con ellas) | `exactitud_bandera`, `metricas_bandera`, `exactitud_oracion`, `puntajes_correccion`, `puntaje_compuesto` | 3 |
| 3 · Ejemplos por oración | `armar_pares_oracion_nota`, `calcular_peso_clase` | 4 |
| 4 · Detector BiomedBERT | `tokenizar_pares`, `entrenar_detector`, `puntuar_oraciones` | 5 |
| 5 · Decisión A y B | `ajustar_umbral`, `decidir_bandera`, `elegir_oracion` | 6 |
| 6 · Corrector BioBART | `marcar_oracion`, `armar_ejemplos_correccion`, `entrenar_corrector`, `generar_correccion` | 7 |
| 7 · Control de calidad | `controlar_correccion`, `calibrar_umbral_calidad` | 8 |
| 8 · Salida y evaluación | `formatear_salida` y la evaluación final en prueba | 9 |
| Extensión · MedQA | `indexar_medqa`, `recuperar_pregunta`, `candidatos_correccion` | 10 |

**Datos.** Entrenamiento: `datos/MEDIQA-CORR-2024-MS-TrainingData.csv` (2.189 notas). Prueba final: las 574 notas
del conjunto de validación oficial de MS, `datos/MEDEC-MS-ValidationSet-with-GroundTruth-and-ErrorType.csv`,
del repositorio de MEDEC (<https://github.com/abachaa/MEDEC>). La prueba **no se toca** hasta la sección 9:
los umbrales y el mejor punto de cada modelo se eligen con la validación por caso de la sección 2.

In [1]:
import copy
import json
import math
import os
import random
import re
import time
from collections import defaultdict
from dataclasses import dataclass
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupKFold

RAIZ = Path.cwd()
DATOS = RAIZ / "datos"
RESULTADOS = RAIZ / "resultados"
MODELOS = RAIZ / "modelos"
RESULTADOS.mkdir(exist_ok=True)
MODELOS.mkdir(exist_ok=True)

CSV_ENTRENAMIENTO = DATOS / "MEDIQA-CORR-2024-MS-TrainingData.csv"
CSV_PRUEBA = DATOS / "MEDEC-MS-ValidationSet-with-GroundTruth-and-ErrorType.csv"

# TALLER7_RAPIDO=1 corre todo con pocas notas y una época, solo para probar que el código no falla.
RAPIDO = os.environ.get("TALLER7_RAPIDO") == "1"
SEMILLA = 42
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"


def fijar_semilla(semilla: int = SEMILLA) -> None:
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


fijar_semilla()
pd.set_option("display.max_colwidth", 120)

# Hiperparámetros: los mismos de la tabla «Hiperparámetros» del reporte.
HIPER = {
    "detector": {
        "modelo": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "max_length": 512,
        "lote": 16,
        "tasa": 2e-5,
        "epocas_min": 1 if RAPIDO else 3,  # 3–4 épocas: al menos 3 y como mucho 4,
        "epocas_max": 1 if RAPIDO else 4,  # con early stopping por F1 de A a partir de la tercera
        "paciencia": 1,
        "pos_weight": None,  # None = negativos / positivos (calcular_peso_clase)
        "acumulacion": 1,  # pasos de acumulación de gradiente (lote efectivo = lote × acumulación)
        "pista_medqa": False,  # True: la nota va precedida de la respuesta de la pregunta de MedQA recuperada
    },
    "decision": {
        "combinar_medqa": False,  # True: la bandera sale de una regresión logística con rasgos de MedQA
    },
    "corrector": {
        "modelo": "GanjinZero/biobart-v2-large",
        "max_length_entrada": 1024,
        "max_new_tokens": 96,
        "lote": 8,
        "tasa": 3e-5,
        "epocas": 1 if RAPIDO else 5,  # hasta 5, mejor punto por ROUGE-1 en validación
        "num_beams": 4,
        "pista_medqa": None,  # None, "opciones" (las 4 opciones como pista) o "respuesta" (la respuesta de MedQA)
        "reordenar_medqa": False,  # True: reordenar_con_medqa antes del control de calidad
    },
    "particion": {"k": 5},
}

# Para probar otra configuración sin editar el notebook: TALLER7_HIPER='{"detector": {"tasa": 3e-5}}'.
for caja, cambios in json.loads(os.environ.get("TALLER7_HIPER", "{}")).items():
    HIPER[caja].update(cambios)

print("Dispositivo:", DISPOSITIVO, torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "")
print("torch", torch.__version__, "· modo rápido:", RAPIDO)

Dispositivo: cuda NVIDIA GeForce RTX 4060 Ti
torch 2.15.0+cu132 · modo rápido: False


## 1 · Carga y limpieza

| Función | Qué hace |
|---|---|
| `cargar_corpus(ruta) -> DataFrame` | Lee el CSV y normaliza tipos (`Error Sentence ID` a entero, `NA` a `None`) |
| `parsear_oraciones(columna) -> dict[int, str]` | `"0 texto\n1 texto"` → `{0: "texto", 1: "texto"}`. Las líneas sin id se pegan a la oración anterior |
| `unir_fragmentos(oraciones) -> list[Segmento]` | Pega a la oración anterior los fragmentos que no son oración (`"mm"`, `"Hg."`, `"100/68 mm"`) |
| `mapear_ids_oficiales(segmentos) -> dict[int, list[int]]` | Qué ids oficiales cubre cada segmento, para devolver el id que exige la subtarea B |

In [2]:
def cargar_corpus(ruta: Path) -> pd.DataFrame:
    """Lee el CSV de MEDIQA-CORR y normaliza tipos: el id de la oración con error queda entero (-1 sin error)
    y los 'NA' de las columnas de texto quedan como None."""
    df = pd.read_csv(ruta)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    df["Error Flag"] = df["Error Flag"].astype(int)
    df["Error Sentence ID"] = df["Error Sentence ID"].fillna(-1).astype(int)
    for columna in ["Error Sentence", "Corrected Sentence", "Corrected Text"]:
        df[columna] = df[columna].apply(
            lambda v: v.strip() if isinstance(v, str) and v.strip() not in ("", "NA") else None
        )
    return df.reset_index(drop=True)


def parsear_oraciones(columna: str) -> dict[int, str]:
    """'0 texto\\n1 texto' → {0: 'texto', 1: 'texto'}.

    Las líneas sin id (valores de laboratorio como 'Platelet count 100,000') se pegan a la oración anterior:
    traen evidencia clínica y pueden contener el error, así que no se descartan.
    """
    oraciones: dict[int, str] = {}
    ultimo = None
    for linea in columna.split("\n"):
        m = re.match(r"^(\d+) (.*)$", linea)
        if m:
            ultimo = int(m.group(1))
            oraciones[ultimo] = m.group(2).strip()
        elif linea.strip() and ultimo is not None:
            oraciones[ultimo] += "\n" + linea.strip()
    return oraciones


@dataclass
class Segmento:
    """Una o varias oraciones oficiales unidas: el texto que lee el modelo y los ids que cubre."""

    texto: str
    ids: list[int]
    partes: list[str]


FIN_DE_ORACION = re.compile(r"[.!?:]\s*$")
UNIDAD = re.compile(r"^(mm|Hg|H2O|mmHg|L\b|dL|mg|g/|U/L|mEq|/|\+|-)")


def es_fragmento(texto: str, previo: str | None) -> bool:
    """Un fragmento continúa la oración anterior: la anterior no terminó en . ! ? : y el fragmento empieza
    con minúscula, número o símbolo, o es una unidad ('Hg.', 'mmHg, pulse is', 'L\\nK+')."""
    if previo is None or FIN_DE_ORACION.search(previo) or not texto:
        return False
    primera = texto.split()[0]
    return not primera[0].isupper() or bool(UNIDAD.match(primera))


def unir_fragmentos(oraciones: dict[int, str]) -> list[Segmento]:
    """Pega cada fragmento a la oración anterior para que el modelo lea frases completas
    ('blood pressure is 100/68 mm' + 'Hg.')."""
    segmentos: list[Segmento] = []
    previo = None
    for i in sorted(oraciones):
        texto = oraciones[i]
        if segmentos and es_fragmento(texto, previo):
            s = segmentos[-1]
            s.texto += " " + texto
            s.ids.append(i)
            s.partes.append(texto)
        else:
            segmentos.append(Segmento(texto=texto, ids=[i], partes=[texto]))
        previo = texto
    return segmentos


def mapear_ids_oficiales(segmentos: list[Segmento]) -> dict[int, list[int]]:
    """Segmento → ids oficiales que cubre. El primero de la lista es el id que se devuelve en la subtarea B:
    el de la parte más larga, que es la oración real y no la unidad suelta."""
    mapa = {}
    for k, s in enumerate(segmentos):
        orden = sorted(range(len(s.ids)), key=lambda j: -len(s.partes[j].split()))
        mapa[k] = [s.ids[j] for j in orden]
    return mapa


def preparar(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica la caja 1 a cada nota."""
    df = df.copy()
    df["oraciones"] = df["Sentences"].apply(parsear_oraciones)
    df["segmentos"] = df["oraciones"].apply(unir_fragmentos)
    df["mapa_ids"] = df["segmentos"].apply(mapear_ids_oficiales)
    return df


entrenamiento = preparar(cargar_corpus(CSV_ENTRENAMIENTO))
prueba = preparar(cargar_corpus(CSV_PRUEBA))
print(f"Entrenamiento: {len(entrenamiento)} notas ({entrenamiento['Error Flag'].sum()} con error)")
print(f"Prueba (validación oficial MS): {len(prueba)} notas ({prueba['Error Flag'].sum()} con error)")

Entrenamiento: 2189 notas (1219 con error)
Prueba (validación oficial MS): 574 notas (319 con error)


In [3]:
# Ejemplo: una nota con fragmentos antes y después de unirlos.
ejemplo = entrenamiento[entrenamiento["Sentences"].str.contains(r"\d+ Hg\.")].iloc[0]
print("Oraciones oficiales:")
for i, t in list(ejemplo["oraciones"].items())[:8]:
    print(f"  {i:>2} {t!r}")
print("\nSegmentos para el modelo (ids que cubre):")
for s in ejemplo["segmentos"][:6]:
    print(f"  {s.ids} {s.texto!r}")

Oraciones oficiales:
   0 'A 53-year-old man comes to the physician because of a 1-day history of fever and chills, severe malaise, and cough with yellow-green sputum.'
   1 'He works as a commercial fisherman on Lake Superior.'
   2 'Current medications include metoprolol and warfarin.'
   3 'His temperature is 38.5 C (101.3 F), pulse is 96/min, respirations are 26/min, and blood pressure is 98/62 mm'
   4 'Hg.'
   5 'Examination shows increased fremitus and bronchial breath sounds over the right middle lung field.'
   6 'After reviewing imaging, the causal pathogen was determined to be Haemophilus influenzae.'
   7 'An x-ray of the chest showed consolidation of the right upper lobe.'

Segmentos para el modelo (ids que cubre):
  [0] 'A 53-year-old man comes to the physician because of a 1-day history of fever and chills, severe malaise, and cough with yellow-green sputum.'
  [1] 'He works as a commercial fisherman on Lake Superior.'
  [2] 'Current medications include metoprolol and wa

In [4]:
def resumen_limpieza(df: pd.DataFrame) -> dict:
    """Cuánto cambia la segmentación y cuánto se pierde en B al devolver el id oficial."""
    oraciones = int(df["oraciones"].str.len().sum())
    segmentos = int(df["segmentos"].str.len().sum())
    errores = df[df["Error Flag"] == 1]
    techo = []
    for _, nota in errores.iterrows():
        k = next(k for k, s in enumerate(nota["segmentos"]) if nota["Error Sentence ID"] in s.ids)
        techo.append(nota["mapa_ids"][k][0] == nota["Error Sentence ID"])
    return {
        "oraciones_oficiales": oraciones,
        "segmentos": segmentos,
        "fragmentos_unidos": oraciones - segmentos,
        "lineas_sin_id_pegadas": int(df["Sentences"].apply(
            lambda s: sum(1 for x in s.split("\n") if x.strip() and not re.match(r"^\d+ ", x))).sum()),
        "techo_B_tras_mapear_ids": round(float(np.mean(techo)), 4),
    }


limpieza = {"entrenamiento": resumen_limpieza(entrenamiento), "prueba": resumen_limpieza(prueba)}
pd.DataFrame(limpieza)

,entrenamiento,prueba
oraciones_oficiales,27673.0000,7162.0000
segmentos,24276.0000,6286.0000
fragmentos_unidos,3397.0000,876.0000
lineas_sin_id_pegadas,1131.0000,252.0000
techo_B_tras_mapear_ids,0.9984,0.9969


El **techo de B** dice qué fracción de las notas con error recupera el id oficial correcto si el detector acierta
el segmento. Unir fragmentos cuesta muy poco en B y le da al modelo frases completas.

## 2 · Partición por caso

| Función | Qué hace |
|---|---|
| `identificar_casos(textos, oraciones) -> list[int]` | Une dos notas en un caso si comparten al menos una oración y su texto coincide en ≥ 0,7 (`SequenceMatcher` por palabras) |
| `dividir_por_caso(df, k=5, semilla) -> list[tuple[índices, índices]]` | `GroupKFold` por caso: las dos versiones de un caso caen siempre del mismo lado |

In [5]:
SIMILITUD_MISMO_CASO = 0.7  # la distribución de similitud tiene un hueco entre 0,4 y 0,8


def identificar_casos(textos: list[str], oraciones: list[dict[int, str]]) -> list[int]:
    """Une las notas del mismo caso: comparten al menos una oración y su texto se parece en
    ≥ SIMILITUD_MISMO_CASO. La primera oración sola no sirve de clave: junta casos distintos que abren igual y
    separa los pares cuyo error está en la oración 0."""
    padre = list(range(len(textos)))

    def raiz(i: int) -> int:
        while padre[i] != i:
            padre[i] = padre[padre[i]]
            i = padre[i]
        return i

    indice = defaultdict(set)
    for i, o in enumerate(oraciones):
        for oracion in o.values():
            indice[oracion].add(i)
    candidatos = set()
    for notas in indice.values():
        if len(notas) < 300:  # oraciones genéricas ('Vital signs are within normal limits.')
            notas = sorted(notas)
            candidatos |= {(a, b) for k, a in enumerate(notas) for b in notas[k + 1:]}
    palabras = [t.split() for t in textos]
    for a, b in candidatos:
        if SequenceMatcher(None, palabras[a], palabras[b], autojunk=False).ratio() >= SIMILITUD_MISMO_CASO:
            padre[raiz(a)] = raiz(b)
    return [raiz(i) for i in range(len(textos))]


def dividir_por_caso(df: pd.DataFrame, k: int = 5, semilla: int = SEMILLA) -> list[tuple[np.ndarray, np.ndarray]]:
    """GroupKFold por caso: devuelve k pares (índices de entrenamiento, índices de validación)."""
    gkf = GroupKFold(n_splits=k, shuffle=True, random_state=semilla)
    return list(gkf.split(df, groups=df["caso"]))


entrenamiento["caso"] = identificar_casos(entrenamiento["Text"].tolist(), entrenamiento["oraciones"].tolist())
tam = entrenamiento.groupby("caso").agg(n=("Error Flag", "size"), errores=("Error Flag", "sum"))
print(f"Casos: {len(tam)} · pares correcta + error: {((tam.n == 2) & (tam.errores == 1)).sum()}"
      f" · solo versión con error: {((tam.n == 1) & (tam.errores == 1)).sum()}"
      f" · grupos de más de 2 notas: {(tam.n > 2).sum()}")

# ¿Algún caso de entrenamiento es también un caso de la prueba? Se buscan con la misma regla sobre la unión.
union = pd.concat([entrenamiento.assign(lado="entrenamiento"), prueba.assign(lado="prueba")], ignore_index=True)
union["caso_union"] = identificar_casos(union["Text"].tolist(), union["oraciones"].tolist())
cruzados = union.groupby("caso_union")["lado"].nunique().loc[lambda s: s > 1].index
fuga = union[(union["caso_union"].isin(cruzados)) & (union["lado"] == "entrenamiento")]["Text ID"]
print(f"Casos compartidos entre entrenamiento y prueba: {len(cruzados)} → se sacan {len(fuga)} notas de entrenamiento")
entrenamiento = entrenamiento[~entrenamiento["Text ID"].isin(fuga)].reset_index(drop=True)

pliegues = dividir_por_caso(entrenamiento, k=HIPER["particion"]["k"])
idx_train, idx_val = pliegues[0]
train = entrenamiento.iloc[idx_train].reset_index(drop=True)
val = entrenamiento.iloc[idx_val].reset_index(drop=True)
assert not set(train["caso"]) & set(val["caso"]), "un caso quedó a los dos lados"

if RAPIDO:
    train, val, prueba = train.sample(150, random_state=0), val.sample(60, random_state=0), prueba.sample(40, random_state=0)
    train, val, prueba = (d.reset_index(drop=True) for d in (train, val, prueba))

pd.DataFrame({
    "notas": [len(train), len(val), len(prueba)],
    "con error": [train["Error Flag"].sum(), val["Error Flag"].sum(), prueba["Error Flag"].sum()],
    "casos": [train["caso"].nunique(), val["caso"].nunique(), "—"],
}, index=["entrenamiento (pliegue 1 de 5)", "validación (pliegue 1 de 5)", "prueba (validación oficial MS)"])

Casos: 1219 · pares correcta + error: 967 · solo versión con error: 250 · grupos de más de 2 notas: 1


Casos compartidos entre entrenamiento y prueba: 2 → se sacan 4 notas de entrenamiento


,notas,con error,casos
entrenamiento (pliegue 1 de 5),1737,973,973
validación (pliegue 1 de 5),448,244,244
prueba (validación oficial MS),574,319,—


Usamos el **primer pliegue** de los cinco: el 80 % de los casos para entrenar y el 20 % para validar (elegir la
época, el umbral de la bandera y el del control de calidad). Ninguna nota de un caso de validación está en
entrenamiento.

## 2b · Extensión MedQA: recuperar la pregunta de cada nota

| Función | Qué hace |
|---|---|
| `indexar_medqa(dataset)` | Índice TF-IDF de MedQA-USMLE de 4 opciones (`GBaker/MedQA-USMLE-4-options`; su partición de validación, de la versión `-hf`) |
| `recuperar_pregunta(nota, k=3) -> list[Pregunta]` | Las preguntas de MedQA más parecidas a la nota |
| `candidatos_correccion(pregunta) -> list[str]` | Las opciones de respuesta, pistas para el corrector o para reordenar sus haces |

En el plan esta extensión solo alimentaba al corrector. La traemos aquí, antes de las cajas 3–7, porque los
experimentos (sección 11) mostraron que la subtarea A no se resuelve sin ella. Cada nota de MS salió de una
pregunta de MedQA, así que recuperar esa pregunta da sus 4 opciones y su respuesta correcta. Solo sirve en MS.

In [6]:
from datasets import load_dataset
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import linear_kernel


@dataclass
class Pregunta:
    texto: str
    opciones: dict[str, str]
    respuesta: str
    similitud: float


class IndiceMedQA:
    def __init__(self, preguntas: list[dict]):
        self.preguntas = preguntas
        self.vectorizador = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, stop_words="english")
        self.matriz = self.vectorizador.fit_transform([p["question"] for p in preguntas])


def cargar_medqa() -> list[dict]:
    """MedQA-USMLE de 4 opciones completo. `GBaker/MedQA-USMLE-4-options` solo trae train y test; la partición de
    validación (1.272 preguntas) está en la versión `-hf` del mismo dataset, con otro formato."""
    base = load_dataset("GBaker/MedQA-USMLE-4-options")
    preguntas = [{"question": q, "options": o, "answer": a} for s in base
                 for q, o, a in zip(base[s]["question"], base[s]["options"], base[s]["answer"])]
    for r in load_dataset("GBaker/MedQA-USMLE-4-options-hf")["validation"]:
        opciones = {letra: r[f"ending{i}"] for i, letra in enumerate("ABCD")}
        preguntas.append({"question": f"{r['sent1']} {r['sent2']}".strip(), "options": opciones,
                          "answer": opciones["ABCD"[r["label"]]]})
    return preguntas


def indexar_medqa(dataset: list[dict]) -> IndiceMedQA:
    """Índice TF-IDF (unigramas y bigramas) sobre el enunciado de todas las preguntas de MedQA."""
    return IndiceMedQA(list(dataset))


def recuperar_pregunta(nota: str, k: int = 3, indice: IndiceMedQA | None = None) -> list[Pregunta]:
    indice = indice or INDICE_MEDQA
    sim = linear_kernel(indice.vectorizador.transform([nota]), indice.matriz).ravel()
    mejores = np.argsort(-sim)[:k]
    return [Pregunta(texto=indice.preguntas[i]["question"], opciones=indice.preguntas[i]["options"],
                     respuesta=indice.preguntas[i]["answer"], similitud=float(sim[i])) for i in mejores]


def candidatos_correccion(pregunta: Pregunta) -> list[str]:
    return [o for o in pregunta.opciones.values() if o]


def sustituir_opcion(original: str, pregunta: Pregunta) -> str | None:
    """Si la oración trae literalmente una opción incorrecta, la cambia por la respuesta correcta (respetando la
    mayúscula inicial). None si no trae ninguna."""
    for opcion in sorted(candidatos_correccion(pregunta), key=len, reverse=True):
        if opcion == pregunta.respuesta:
            continue
        patron = re.compile(r"(?<![A-Za-z])" + re.escape(opcion) + r"(?![A-Za-z])", re.IGNORECASE)
        encontrado = patron.search(original)
        if encontrado:
            r = pregunta.respuesta
            r = r[:1].upper() + r[1:] if encontrado.group(0)[:1].isupper() else r[:1].lower() + r[1:]
            return original[:encontrado.start()] + r + original[encontrado.end():]
    return None


def reordenar_con_medqa(nota: pd.Series, original: str, haces: list[str]) -> list[str]:
    """Reordena los haces del corrector con la pregunta recuperada: primero la sustitución de la opción incorrecta
    por la respuesta (si aplica), después el primer haz que trae una opción nueva y luego el resto."""
    nuevas = [o.lower() for o in candidatos_correccion(nota["medqa"]) if o.lower() not in original.lower()]
    con_opcion = [h for h in haces if any(o in h.lower() for o in nuevas)]
    orden = con_opcion[:1] + [h for h in haces if h not in con_opcion[:1]]
    sustitucion = sustituir_opcion(original, nota["medqa"])
    return ([sustitucion] if sustitucion else []) + [h for h in orden if h != sustitucion]


PALABRA = re.compile(r"[a-z0-9]+")


def _palabras(texto: str) -> set[str]:
    return {w for w in PALABRA.findall(texto.lower()) if w not in ENGLISH_STOP_WORDS}


def contencion(frase: str, texto: str) -> float:
    """Fracción de las palabras de contenido de `frase` que aparecen en `texto` (1 = todas)."""
    palabras = _palabras(frase)
    return len(palabras & _palabras(texto)) / len(palabras) if palabras else 0.0


def rasgos_medqa(nota: pd.Series) -> dict:
    """¿La nota trae la respuesta correcta de su pregunta, o una de las otras opciones?"""
    p = nota["medqa"]
    correcta = contencion(p.respuesta, nota["Text"])
    otras = max((contencion(o, nota["Text"]) for o in candidatos_correccion(p) if o != p.respuesta), default=0.0)
    return {"similitud": p.similitud, "contencion_respuesta": correcta, "contencion_otras": otras,
            "diferencia": correcta - otras}


INDICE_MEDQA = indexar_medqa(cargar_medqa())
print(f"Preguntas indexadas: {len(INDICE_MEDQA.preguntas)}")
for d in (train, val, prueba):
    d["medqa"] = [recuperar_pregunta(t, k=1)[0] for t in d["Text"]]
    rasgos = pd.DataFrame([rasgos_medqa(n) for _, n in d.iterrows()], index=d.index)
    d[rasgos.columns] = rasgos

# ¿Hace falta la partición de validación de MedQA? Misma recuperación con y sin ella, sobre nuestra validación.
sin_dev = indexar_medqa([p for p in INDICE_MEDQA.preguntas[:-1272]])
sim_sin_dev = np.array([recuperar_pregunta(t, k=1, indice=sin_dev)[0].similitud for t in val["Text"]])
print(f"Notas de validación cuya pregunta no se encuentra (similitud < 0,5): sin la partición de validación de MedQA "
      f"{np.mean(sim_sin_dev < 0.5):.1%}, con ella {np.mean(val['similitud'] < 0.5):.1%}")
del sin_dev

# ¿Cuánto dice MedQA de la subtarea A? Solo con validación.
correcta_literal = val.apply(lambda n: n["medqa"].respuesta.lower() in n["Text"].lower(), axis=1)
pd.Series({
    "similitud de la pregunta recuperada (mediana)": val["similitud"].median(),
    "notas con similitud < 0,5 (la pregunta no se encontró)": (val["similitud"] < 0.5).mean(),
    "notas correctas que traen literal la respuesta de MedQA": correcta_literal[val["Error Flag"] == 0].mean(),
    "notas con error que traen literal la respuesta de MedQA": correcta_literal[val["Error Flag"] == 1].mean(),
    "AUC de «diferencia» para separar notas correctas de notas con error": roc_auc_score(1 - val["Error Flag"], val["diferencia"]),
}).to_frame("validación")

/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Preguntas indexadas: 12723


Notas de validación cuya pregunta no se encuentra (similitud < 0,5): sin la partición de validación de MedQA 16.5%, con ella 5.8%


,validación
similitud de la pregunta recuperada (mediana),0.952462
"notas con similitud < 0,5 (la pregunta no se encontró)",0.058036
notas correctas que traen literal la respuesta de MedQA,0.789216
notas con error que traen literal la respuesta de MedQA,0.008197
AUC de «diferencia» para separar notas correctas de notas con error,0.972437


## 3 · Métricas (caja 8)

Las métricas de la caja 8 se definen aquí porque cada caja se compara con ellas. Replican el script oficial de
MEDIQA-CORR 2024 (`mediqa-corr-2024-eval-script-2-all-metrics.py`): ROUGE-1 F del paquete `rouge`, BERTScore con
`microsoft/deberta-xlarge-mnli` reescalado con la línea base, y BLEURT-20, los dos últimos recortados a [0, 1].
BLEURT-20 corre con su versión en PyTorch (`lucadiliello/BLEURT-20`).

| Función | Qué hace |
|---|---|
| `exactitud_bandera(pred, real) -> float` | Subtarea A (métrica oficial) |
| `metricas_bandera(pred, real) -> dict` | Precisión, *recall*, F1 y matriz de confusión de la bandera |
| `exactitud_oracion(pred, real) -> float` | Subtarea B (acertar `-1` también cuenta); se aplica aparte a las notas con error |
| `puntajes_correccion(pred, ref) -> dict` | ROUGE-1, BERTScore, BLEURT-20 y su promedio, solo donde ambas son oraciones |
| `puntaje_compuesto(pred, ref) -> float` | Regla oficial: 1 si ambas son `NA`, 0 si solo una lo es y la métrica en otro caso |

In [7]:
from rouge import Rouge

NA = "NA"


def exactitud_bandera(pred: list[int], real: list[int]) -> float:
    return float(np.mean(np.asarray(pred) == np.asarray(real)))


def metricas_bandera(pred: list[int], real: list[int]) -> dict:
    p, r = np.asarray(pred), np.asarray(real)
    vp, fp = int(((p == 1) & (r == 1)).sum()), int(((p == 1) & (r == 0)).sum())
    fn, vn = int(((p == 0) & (r == 1)).sum()), int(((p == 0) & (r == 0)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "matriz": {"verdaderos_positivos": vp, "falsos_positivos": fp,
                       "falsos_negativos": fn, "verdaderos_negativos": vn}}


def exactitud_oracion(pred: list[int], real: list[int]) -> float:
    return float(np.mean(np.asarray(pred) == np.asarray(real)))


def _limpiar(texto: str) -> str:
    return re.sub(r"\s+", " ", texto.replace("\n", " ")).strip().strip('"').strip()


class MetricasCorreccion:
    """Carga BERTScore y BLEURT-20 una sola vez y guarda en caché el puntaje de cada par (predicción, referencia)."""

    def __init__(self, dispositivo: str = DISPOSITIVO):
        self.dispositivo = dispositivo
        self.cache: dict[tuple[str, str], dict] = {}
        self._bertscore = None
        self._bleurt = None

    def _cargar(self):
        if self._bertscore is None:
            from bert_score import BERTScorer
            from bleurt_pytorch import BleurtForSequenceClassification, BleurtTokenizer

            self._bertscore = BERTScorer(model_type="microsoft/deberta-xlarge-mnli", lang="en",
                                         rescale_with_baseline=True, device=self.dispositivo)
            self._bleurt_tok = BleurtTokenizer.from_pretrained("lucadiliello/BLEURT-20")
            self._bleurt = BleurtForSequenceClassification.from_pretrained("lucadiliello/BLEURT-20")
            self._bleurt = self._bleurt.to(self.dispositivo).eval()

    def liberar(self) -> None:
        """Saca BERTScore y BLEURT de la GPU (se vuelven a cargar solos); la caché de puntajes se conserva."""
        self._bertscore = self._bleurt = None
        torch.cuda.empty_cache()

    @staticmethod
    def _rouge1(pred: str, ref: str) -> float:
        try:
            return Rouge().get_scores(pred, ref)[0]["rouge-1"]["f"]
        except ValueError:  # predicción vacía
            return 0.0

    @torch.no_grad()
    def _bleurt_lote(self, preds: list[str], refs: list[str], lote: int = 32) -> list[float]:
        salida = []
        for i in range(0, len(preds), lote):
            enc = self._bleurt_tok(refs[i:i + lote], preds[i:i + lote], padding="longest", truncation=True,
                                   max_length=512, return_tensors="pt").to(self.dispositivo)
            salida += self._bleurt(**enc).logits.flatten().float().cpu().tolist()
        return salida

    def puntuar(self, preds: list[str], refs: list[str]) -> list[dict]:
        pares = [(_limpiar(p), _limpiar(r)) for p, r in zip(preds, refs)]
        nuevos = sorted({par for par in pares if par not in self.cache})
        if nuevos:
            self._cargar()
            p_nuevos, r_nuevos = [p for p, _ in nuevos], [r for _, r in nuevos]
            _, _, f1 = self._bertscore.score(p_nuevos, r_nuevos, batch_size=32)
            bleurt = self._bleurt_lote(p_nuevos, r_nuevos)
            for (p, r), bs, bl in zip(nuevos, f1.tolist(), bleurt):
                d = {"rouge1": self._rouge1(p, r), "bertscore": min(1.0, max(0.0, bs)),
                     "bleurt": min(1.0, max(0.0, bl))}
                d["agregado"] = (d["rouge1"] + d["bertscore"] + d["bleurt"]) / 3
                self.cache[(p, r)] = d
        return [self.cache[par] for par in pares]


METRICAS = MetricasCorreccion()


def puntajes_correccion(pred: list[str], ref: list[str]) -> dict:
    """ROUGE-1, BERTScore, BLEURT-20 y su promedio (AggregateScore), solo sobre pares donde ambas son oraciones."""
    por_par = METRICAS.puntuar(pred, ref)
    medias = {m: float(np.mean([d[m] for d in por_par])) if por_par else 0.0
              for m in ["rouge1", "bertscore", "bleurt", "agregado"]}
    return {**medias, "por_par": por_par}


def puntaje_compuesto(pred: list[str | None], ref: list[str | None], metrica: str = "agregado") -> float:
    """Regla oficial: por nota, 1 si ambas son NA, 0 si solo una lo es y la métrica en otro caso; promedio
    sobre todas las notas."""
    pred = [p if isinstance(p, str) else NA for p in pred]  # None o NaN (pandas) cuentan como NA
    ref = [r if isinstance(r, str) else NA for r in ref]
    ambas_na = sum(p == NA and r == NA for p, r in zip(pred, ref))
    pares = [(p, r) for p, r in zip(pred, ref) if p != NA and r != NA]
    suma = sum(d[metrica] for d in METRICAS.puntuar([p for p, _ in pares], [r for _, r in pares])) if pares else 0.0
    return (suma + ambas_na) / len(ref)

### Las dos bases que el pipeline tiene que superar

Siempre «hay error» en A y siempre «la última oración» en B, medidas en entrenamiento (como en el reporte),
en nuestra validación y en la prueba.

In [8]:
def bases(df: pd.DataFrame) -> dict:
    con_error = df[df["Error Flag"] == 1]
    ultima = con_error["oraciones"].apply(lambda o: max(o))
    return {
        "A · siempre «hay error»": exactitud_bandera([1] * len(df), df["Error Flag"]),
        "B · última oración (notas con error)": exactitud_oracion(ultima, con_error["Error Sentence ID"]),
    }


tabla_bases = pd.DataFrame({"entrenamiento": bases(entrenamiento), "validación": bases(val), "prueba": bases(prueba)})
tabla_bases.style.format("{:.1%}")

,entrenamiento,validación,prueba
A · siempre «hay error»,55.7%,54.5%,55.6%
B · última oración (notas con error),62.3%,60.2%,61.8%


## 4 · Ejemplos por oración (caja 3)

| Función | Qué hace |
|---|---|
| `armar_pares_oracion_nota(nota) -> list[Par]` | Por cada segmento produce `(oración candidata, nota completa, etiqueta)`, con etiqueta 1 solo si el segmento contiene el id de error |
| `calcular_peso_clase(pares) -> float` | `negativos / positivos ≈ 21,7` para `pos_weight` de la pérdida |

In [9]:
@dataclass
class Par:
    text_id: str
    indice_segmento: int
    oracion: str
    nota: str
    etiqueta: int


def armar_pares_oracion_nota(nota: pd.Series) -> list[Par]:
    """Un par por segmento: la oración candidata va con la nota completa, porque el error solo se ve en contexto.
    Con pista_medqa, la nota va precedida de la respuesta correcta de su pregunta de MedQA."""
    texto = nota["Text"]
    if HIPER["detector"]["pista_medqa"]:
        texto = f"MedQA answer: {nota['medqa'].respuesta}. {texto}"
    return [
        Par(text_id=nota["Text ID"], indice_segmento=k, oracion=s.texto, nota=texto,
            etiqueta=int(nota["Error Flag"] == 1 and nota["Error Sentence ID"] in s.ids))
        for k, s in enumerate(nota["segmentos"])
    ]


def calcular_peso_clase(pares: list[Par]) -> float:
    positivos = sum(p.etiqueta for p in pares)
    return (len(pares) - positivos) / positivos


def armar_pares(df: pd.DataFrame) -> list[Par]:
    return [par for _, nota in df.iterrows() for par in armar_pares_oracion_nota(nota)]


pares_train, pares_val, pares_prueba = armar_pares(train), armar_pares(val), armar_pares(prueba)
PESO_CLASE = HIPER["detector"]["pos_weight"] or calcular_peso_clase(pares_train)
print(f"Pares de entrenamiento: {len(pares_train)} ({sum(p.etiqueta for p in pares_train)} positivos)")
print(f"Pares de validación: {len(pares_val)} · pares de prueba: {len(pares_prueba)}")
print(f"pos_weight = negativos / positivos = {PESO_CLASE:.1f} "
      f"(21,7 en el reporte con las oraciones oficiales; aquí baja porque los fragmentos unidos ya no cuentan)")
pares_train[0]

Pares de entrenamiento: 19294 (973 positivos)
Pares de validación: 4942 · pares de prueba: 6286
pos_weight = negativos / positivos = 18.8 (21,7 en el reporte con las oraciones oficiales; aquí baja porque los fragmentos unidos ya no cuentan)


Par(text_id='ms-train-0', indice_segmento=0, oracion='A 53-year-old man comes to the physician because of a 1-day history of fever and chills, severe malaise, and cough with yellow-green sputum.', nota='A 53-year-old man comes to the physician because of a 1-day history of fever and chills, severe malaise, and cough with yellow-green sputum. He works as a commercial fisherman on Lake Superior. Current medications include metoprolol and warfarin. His temperature is 38.5 C (101.3 F), pulse is 96/min, respirations are 26/min, and blood pressure is 98/62 mm Hg. Examination shows increased fremitus and bronchial breath sounds over the right middle lung field. After reviewing imaging, the causal pathogen was determined to be Haemophilus influenzae. An x-ray of the chest showed consolidation of the right upper lobe.', etiqueta=0)

## 5 · Detector BiomedBERT (caja 4)

| Función | Qué hace |
|---|---|
| `tokenizar_pares(pares, tokenizador) -> BatchEncoding` | `[CLS] oración [SEP] nota [SEP]`, `truncation="only_second"`, `max_length=512` |
| `entrenar_detector(pares_train, pares_val) -> modelo` | `AutoModelForSequenceClassification(num_labels=1)` con `BCEWithLogitsLoss(pos_weight)`, AdamW 2e-5, lote 16, 3–4 épocas y *early stopping* según el F1 de A por nota en validación |
| `puntuar_oraciones(modelo, nota) -> list[float]` | Probabilidad de error de cada oración de una nota |

El relleno es dinámico: cada lote se rellena hasta su par más largo, no hasta 512. La tasa decae linealmente
después de un calentamiento del 10 % de los pasos.

In [10]:
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)

TOK_DETECTOR = AutoTokenizer.from_pretrained(HIPER["detector"]["modelo"])


def tokenizar_pares(pares: list[Par], tokenizador=None):
    """[CLS] oración [SEP] nota [SEP]; si no cabe, se corta la nota, nunca la oración candidata."""
    tokenizador = tokenizador or TOK_DETECTOR
    return tokenizador([p.oracion for p in pares], [p.nota for p in pares], truncation="only_second",
                       max_length=HIPER["detector"]["max_length"])


def _cargador(pares: list[Par], lote: int, barajar: bool, tokenizador=None) -> DataLoader:
    tokenizador = tokenizador or TOK_DETECTOR
    enc = tokenizar_pares(pares, tokenizador)
    filas = [{**{k: enc[k][i] for k in enc.keys()}, "labels": float(p.etiqueta)} for i, p in enumerate(pares)]
    return DataLoader(filas, batch_size=lote, shuffle=barajar, collate_fn=DataCollatorWithPadding(tokenizador))


@torch.no_grad()
def _probabilidades(modelo, pares: list[Par], lote: int = 64) -> np.ndarray:
    modelo.eval()
    salida = []
    for b in _cargador(pares, lote, barajar=False, tokenizador=getattr(modelo, "tokenizador", None)):
        b = {k: v.to(DISPOSITIVO) for k, v in b.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DISPOSITIVO == "cuda"):
            salida.append(torch.sigmoid(modelo(**b).logits.squeeze(-1).float()).cpu().numpy())
    return np.concatenate(salida)


def probs_por_nota(pares: list[Par], probs: np.ndarray) -> dict[str, list[float]]:
    """Agrupa las probabilidades de los pares por nota, en el orden de sus segmentos."""
    agrupado = defaultdict(list)
    for par, p in zip(pares, probs):
        agrupado[par.text_id].append(float(p))
    return dict(agrupado)


def barrer_umbrales(prob_nota: np.ndarray, bandera_real: np.ndarray) -> pd.DataFrame:
    """Exactitud, precisión, recall y F1 de la bandera por nota para cada umbral (la probabilidad de la nota es la
    de su oración más sospechosa)."""
    filas = []
    for u in np.round(np.arange(0.02, 0.99, 0.01), 2):
        pred = (prob_nota >= u).astype(int)
        m = metricas_bandera(pred, bandera_real)
        filas.append({"umbral": u, "exactitud": exactitud_bandera(pred, bandera_real),
                      "precision": m["precision"], "recall": m["recall"], "f1": m["f1"]})
    return pd.DataFrame(filas)


def _f1_bandera(modelo, pares: list[Par]) -> tuple[float, float]:
    """F1 de A por nota con el mejor umbral y, como diagnóstico, el AUC de la probabilidad por nota."""
    from sklearn.metrics import roc_auc_score

    por_nota = probs_por_nota(pares, _probabilidades(modelo, pares))
    reales = defaultdict(int)
    for p in pares:
        reales[p.text_id] = max(reales[p.text_id], p.etiqueta)
    ids = list(por_nota)
    prob, real = np.array([max(por_nota[i]) for i in ids]), np.array([reales[i] for i in ids])
    return float(barrer_umbrales(prob, real)["f1"].max()), float(roc_auc_score(real, prob))


def entrenar_detector(pares_train: list[Par], pares_val: list[Par]):
    """Ajusta BiomedBERT para decir si una oración, leída junto a la nota, es la del error. Entrena al menos
    epocas_min épocas, se queda con la de mayor F1 de A por nota en validación y, desde epocas_min, para si una
    época no mejora."""
    h = HIPER["detector"]
    fijar_semilla()
    modelo = AutoModelForSequenceClassification.from_pretrained(h["modelo"], num_labels=1).to(DISPOSITIVO)
    modelo.tokenizador = AutoTokenizer.from_pretrained(h["modelo"])  # cada modelo lleva su tokenizador
    peso = h["pos_weight"] or calcular_peso_clase(pares_train)
    perdida = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([peso], device=DISPOSITIVO))
    cargador = _cargador(pares_train, h["lote"], barajar=True, tokenizador=modelo.tokenizador)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=h["tasa"], weight_decay=0.01)
    pasos = math.ceil(len(cargador) / h["acumulacion"]) * h["epocas_max"]
    planificador = get_linear_schedule_with_warmup(optimizador, int(0.1 * pasos), pasos)

    historial, mejor_f1, mejor_estado, sin_mejora = [], -1.0, None, 0
    for epoca in range(1, h["epocas_max"] + 1):
        modelo.train()
        inicio, total = time.time(), 0.0
        for paso, b in enumerate(cargador, start=1):
            b = {k: v.to(DISPOSITIVO) for k, v in b.items()}
            etiquetas = b.pop("labels")
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DISPOSITIVO == "cuda"):
                logits = modelo(**b).logits.squeeze(-1)
            loss = perdida(logits.float(), etiquetas)
            (loss / h["acumulacion"]).backward()
            total += loss.item()
            if paso % h["acumulacion"] == 0 or paso == len(cargador):
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                optimizador.step()
                planificador.step()
                optimizador.zero_grad()
        f1, auc = _f1_bandera(modelo, pares_val)
        historial.append({"época": epoca, "pérdida train": total / len(cargador), "F1 de A (val)": f1,
                          "AUC por nota (val)": auc, "minutos": (time.time() - inicio) / 60})
        print(historial[-1])
        if f1 > mejor_f1:
            mejor_f1, sin_mejora = f1, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if epoca >= h["epocas_min"] and sin_mejora >= h["paciencia"]:
                print(f"Early stopping: la época {epoca} no mejoró el F1 de A")
                break
    modelo.load_state_dict(mejor_estado)
    modelo.historial = pd.DataFrame(historial)
    return modelo.eval()


@torch.no_grad()
def puntuar_oraciones(modelo, nota: pd.Series) -> list[float]:
    """Probabilidad de error de cada segmento de una nota."""
    return _probabilidades(modelo, armar_pares_oracion_nota(nota)).tolist()

In [11]:
detector = entrenar_detector(pares_train, pares_val)
detector.save_pretrained(MODELOS / "detector")
detector.tokenizador.save_pretrained(MODELOS / "detector")
detector.historial

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'época': 1, 'pérdida train': 0.8185723491042395, 'F1 de A (val)': 0.7074235807860263, 'AUC por nota (val)': 0.5266493892639023, 'minutos': 4.186227325598399}


{'época': 2, 'pérdida train': 0.44764218200770384, 'F1 de A (val)': 0.6933333333333334, 'AUC por nota (val)': 0.5130785920925747, 'minutos': 4.177945133050283}


{'época': 3, 'pérdida train': 0.3192265286291046, 'F1 de A (val)': 0.6938775510204082, 'AUC por nota (val)': 0.5212552234008357, 'minutos': 4.179592768351237}
Early stopping: la época 3 no mejoró el F1 de A


,época,pérdida train,F1 de A (val),AUC por nota (val),minutos
0,1,0.818572,0.707424,0.526649,4.186227
1,2,0.447642,0.693333,0.513079,4.177945
2,3,0.319227,0.693878,0.521255,4.179593


**Arquitectura con la forma de cada tensor.** Pasamos un lote de 4 pares por el modelo entrenado y leemos la salida
de cada capa. `BertForSequenceClassification` pasa el vector `[CLS]` por el *pooler* preentrenado de BERT
(Dense 768 → 768 con tanh) antes del Dropout + Dense 768 → 1 de la tabla del reporte.

In [12]:
formas = {}
ganchos = [
    detector.bert.embeddings.register_forward_hook(lambda m, i, o: formas.__setitem__("Embedding", tuple(o.shape))),
    detector.bert.encoder.register_forward_hook(lambda m, i, o: formas.__setitem__("12 bloques Transformer", tuple(o[0].shape))),
    detector.bert.pooler.register_forward_hook(lambda m, i, o: formas.__setitem__("Vector [CLS] (pooler)", tuple(o.shape))),
    detector.classifier.register_forward_hook(lambda m, i, o: formas.__setitem__("Dropout + Dense → logit", tuple(o.shape))),
]
lote = next(iter(_cargador(pares_val[:4], 4, barajar=False)))
with torch.no_grad():
    detector(**{k: v.to(DISPOSITIVO) for k, v in lote.items() if k != "labels"})
for g in ganchos:
    g.remove()
cfg = detector.config
print(f"input_ids {tuple(lote['input_ids'].shape)} · vocabulario {cfg.vocab_size} · dimensión {cfg.hidden_size}"
      f" · capas {cfg.num_hidden_layers} · cabezas {cfg.num_attention_heads} × {cfg.hidden_size // cfg.num_attention_heads}")
print(f"Embedding de tokens: {cfg.vocab_size} × {cfg.hidden_size} = {cfg.vocab_size * cfg.hidden_size / 1e6:.1f} M parámetros")
pd.DataFrame({"salida (B, L, d)": formas})

input_ids (4, 242) · vocabulario 30522 · dimensión 768 · capas 12 · cabezas 12 × 64
Embedding de tokens: 30522 × 768 = 23.4 M parámetros


,"salida (B, L, d)"
Embedding,"(4, 242, 768)"
12 bloques Transformer,"(4, 242, 768)"
Vector [CLS] (pooler),"(4, 768)"
Dropout + Dense → logit,"(4, 1)"


## 6 · Decisión de las subtareas A y B (caja 5)

| Función | Qué hace |
|---|---|
| `decidir_bandera(probs, umbral) -> int` | `1` si la oración más sospechosa supera el umbral |
| `elegir_oracion(probs, mapa_ids) -> int` | Id oficial de la oración más sospechosa, o `-1` si la bandera es 0 |
| `ajustar_umbral(probs_val, etiquetas_val) -> float` | Barre umbrales en validación y se queda con el de mayor F1 de la bandera por nota, reportando también exactitud, precisión y *recall* |

In [13]:
def ajustar_umbral(probs_val: list[list[float]], etiquetas_val: list[int]) -> float:
    """El umbral de mayor F1 de la bandera por nota en validación; deja el barrido completo en BARRIDO_BANDERA."""
    global BARRIDO_BANDERA
    BARRIDO_BANDERA = barrer_umbrales(np.array([max(p) for p in probs_val]), np.asarray(etiquetas_val))
    return float(BARRIDO_BANDERA.loc[BARRIDO_BANDERA["f1"].idxmax(), "umbral"])


def decidir_bandera(probs: list[float], umbral: float) -> int:
    return int(max(probs) >= umbral)


def elegir_oracion(probs: list[float], mapa_ids: dict[int, list[int]], bandera: int = 1) -> int:
    """Id oficial del segmento más sospechoso (el de su oración principal), o -1 si la bandera es 0."""
    if bandera == 0:
        return -1
    return mapa_ids[int(np.argmax(probs))][0]


RASGOS_BANDERA = ["prob_detector", "similitud", "contencion_respuesta", "contencion_otras", "diferencia",
                  "respuesta_en_oracion", "otras_en_oracion"]


def tabla_rasgos(df: pd.DataFrame, probs: dict[str, list[float]]) -> pd.DataFrame:
    """Rasgos de la nota para la bandera: la probabilidad del detector, los de MedQA sobre toda la nota y los mismos
    sobre la oración más sospechosa (¿trae la respuesta correcta o una de las otras opciones?)."""
    filas = []
    for _, n in df.iterrows():
        p = probs[n["Text ID"]]
        oracion = n["segmentos"][int(np.argmax(p))].texto
        pregunta = n["medqa"]
        otras = [o for o in candidatos_correccion(pregunta) if o != pregunta.respuesta]
        filas.append({"prob_detector": max(p), "respuesta_en_oracion": contencion(pregunta.respuesta, oracion),
                      "otras_en_oracion": max((contencion(o, oracion) for o in otras), default=0.0)})
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(filas)], axis=1)[RASGOS_BANDERA]


def ajustar_combinacion(df: pd.DataFrame, probs: dict[str, list[float]]):
    """Regresión logística (rasgos estandarizados) que da la probabilidad de error de la nota a partir del detector y
    de MedQA. Se ajusta en validación; devuelve el modelo y las probabilidades fuera de pliegue (GroupKFold por caso
    dentro de validación), que son las que se usan para elegir el umbral y medir sin trampa."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    X, y = tabla_rasgos(df, probs).values, df["Error Flag"].values
    fuera = np.zeros(len(df))
    for a, b in GroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA).split(X, groups=df["caso"]):
        fuera[b] = make_pipeline(StandardScaler(), LogisticRegression()).fit(X[a], y[a]).predict_proba(X[b])[:, 1]
    modelo = make_pipeline(StandardScaler(), LogisticRegression()).fit(X, y)
    return modelo, dict(zip(df["Text ID"], fuera))


def prob_combinada(modelo, df: pd.DataFrame, probs: dict[str, list[float]]) -> dict[str, float]:
    return dict(zip(df["Text ID"], modelo.predict_proba(tabla_rasgos(df, probs).values)[:, 1]))


def decidir_a_y_b(df: pd.DataFrame, probs: dict[str, list[float]], umbral: float,
                  prob_bandera: dict[str, float] | None = None) -> pd.DataFrame:
    """A y B de cada nota. Sin prob_bandera, la bandera compara la oración más sospechosa con el umbral (el plan);
    con prob_bandera (combinación con MedQA), compara esa probabilidad de la nota."""
    salida = df[["Text ID", "Error Flag", "Error Sentence ID"]].copy()
    salida["probs"] = df["Text ID"].map(probs)
    if prob_bandera is None:
        salida["bandera"] = salida["probs"].apply(lambda p: decidir_bandera(p, umbral))
    else:
        salida["bandera"] = df["Text ID"].map(lambda t: decidir_bandera([prob_bandera[t]], umbral))
    salida["id_pred"] = [elegir_oracion(p, m, b) for p, m, b in zip(salida["probs"], df["mapa_ids"], salida["bandera"])]
    salida["id_mas_sospechoso"] = [elegir_oracion(p, m) for p, m in zip(salida["probs"], df["mapa_ids"])]
    return salida


def evaluar_a_y_b(d: pd.DataFrame) -> dict:
    con_error = d[d["Error Flag"] == 1]
    m = metricas_bandera(d["bandera"], d["Error Flag"])
    return {
        "A · exactitud": exactitud_bandera(d["bandera"], d["Error Flag"]),
        "A · precisión": m["precision"], "A · recall": m["recall"], "A · F1": m["f1"],
        "B · exactitud (todas las notas)": exactitud_oracion(d["id_pred"], d["Error Sentence ID"]),
        "B · exactitud (notas con error)": exactitud_oracion(con_error["id_pred"], con_error["Error Sentence ID"]),
        "B · localización si se sabe que hay error": exactitud_oracion(con_error["id_mas_sospechoso"], con_error["Error Sentence ID"]),
    }


probs_val = probs_por_nota(pares_val, _probabilidades(detector, pares_val))
COMBINACION, prob_bandera_val = None, None
if HIPER["decision"]["combinar_medqa"]:
    COMBINACION, prob_bandera_val = ajustar_combinacion(val, probs_val)  # probabilidades fuera de pliegue
    UMBRAL_BANDERA = ajustar_umbral([[prob_bandera_val[i]] for i in val["Text ID"]], val["Error Flag"].tolist())
else:
    UMBRAL_BANDERA = ajustar_umbral([probs_val[i] for i in val["Text ID"]], val["Error Flag"].tolist())
print(f"Umbral de la bandera (mayor F1 en validación): {UMBRAL_BANDERA}")
BARRIDO_BANDERA.iloc[::10].style.format({"exactitud": "{:.1%}", "precision": "{:.1%}", "recall": "{:.1%}", "f1": "{:.3f}"})

Umbral de la bandera (mayor F1 en validación): 0.92


,umbral,exactitud,precision,recall,f1
0,0.020000,54.2%,54.4%,99.6%,0.703
10,0.120000,54.5%,54.5%,99.6%,0.704
20,0.220000,54.7%,54.6%,99.6%,0.705
30,0.320000,54.7%,54.6%,99.6%,0.705
40,0.420000,54.7%,54.6%,99.6%,0.705
50,0.520000,54.7%,54.6%,99.6%,0.705
60,0.620000,54.7%,54.6%,99.6%,0.705
70,0.720000,54.7%,54.6%,99.6%,0.705
80,0.820000,54.7%,54.6%,99.6%,0.705
90,0.920000,55.1%,54.9%,99.6%,0.707


In [14]:
decision_val = decidir_a_y_b(val, probs_val, UMBRAL_BANDERA, prob_bandera_val)
resultado_val_ab = evaluar_a_y_b(decision_val)
pd.Series({**resultado_val_ab, **{f"base {k}": v for k, v in bases(val).items()}}).to_frame("validación").style.format("{:.1%}")

,validación
A · exactitud,55.1%
A · precisión,54.9%
A · recall,99.6%
A · F1,70.7%
B · exactitud (todas las notas),53.6%
B · exactitud (notas con error),96.7%
B · localización si se sabe que hay error,96.7%
base A · siempre «hay error»,54.5%
base B · última oración (notas con error),60.2%


**¿Por qué la bandera y la localización van tan distinto?** En MS las dos versiones de un caso son iguales salvo en
la oración donde los anotadores pusieron la respuesta del examen (la correcta o una incorrecta). Si el detector
aprendió a encontrar *esa* oración pero no a juzgar si la respuesta es correcta, apuntará a la misma oración en las
dos versiones y con probabilidades parecidas. Lo medimos con los pares de validación (una versión correcta y otra
con error del mismo caso) y con el AUC de la probabilidad por nota (0,5 = no separa notas con y sin error).

In [15]:
from sklearn.metrics import roc_auc_score

prob_nota_val = val["Text ID"].map(lambda t: max(probs_val[t]))
pares_caso = val.assign(prob=prob_nota_val, mas_sospechoso=decision_val["id_mas_sospechoso"]).groupby("caso").filter(
    lambda g: len(g) == 2 and g["Error Flag"].sum() == 1)
correcta = pares_caso[pares_caso["Error Flag"] == 0].set_index("caso")
con_error = pares_caso[pares_caso["Error Flag"] == 1].set_index("caso")
diagnostico = {
    "AUC de la probabilidad por nota (validación)": roc_auc_score(val["Error Flag"], prob_nota_val),
    "pares correcta + error en validación": len(correcta),
    "en la versión correcta, la oración más sospechosa es la que lleva el error en la otra versión":
        float((correcta["mas_sospechoso"] == con_error.loc[correcta.index, "Error Sentence ID"]).mean()),
    "probabilidad media de la nota, versión con error": float(con_error["prob"].mean()),
    "probabilidad media de la nota, versión correcta": float(correcta["prob"].mean()),
    "la versión con error recibe más probabilidad que su versión correcta": float(
        (con_error.loc[correcta.index, "prob"].values > correcta["prob"].values).mean()),
}
mejor_exactitud = BARRIDO_BANDERA.loc[BARRIDO_BANDERA["exactitud"].idxmax()]
print(f"Solo como referencia: el umbral de mayor exactitud en validación sería {mejor_exactitud['umbral']} "
      f"(exactitud {mejor_exactitud['exactitud']:.1%}, F1 {mejor_exactitud['f1']:.3f}); el pipeline usa el de mayor F1, como dice el plan.")
pd.Series(diagnostico).to_frame("validación")

Solo como referencia: el umbral de mayor exactitud en validación sería 0.92 (exactitud 55.1%, F1 0.707); el pipeline usa el de mayor F1, como dice el plan.


,validación
AUC de la probabilidad por nota (validación),0.526649
pares correcta + error en validación,204.000000
"en la versión correcta, la oración más sospechosa es la que lleva el error en la otra versión",0.044118
"probabilidad media de la nota, versión con error",0.974433
"probabilidad media de la nota, versión correcta",0.970373
la versión con error recibe más probabilidad que su versión correcta,0.436275


**Lectura.** La hipótesis de arriba no se cumple: en la versión correcta, la oración más sospechosa coincide con la
del error de la otra versión solo en el 4 % de los pares. El detector sí localiza bien cuando hay error (96,7 % de
las notas con error), pero en **toda** nota encuentra alguna oración con probabilidad alta: la media de la nota es
0,974 con error y 0,970 sin error, y la versión con error gana a su versión correcta solo en el 44 % de los pares.
Por eso el AUC por nota es 0,53 y el umbral de mayor F1 (0,92) marca casi todo: la exactitud de A (55,1 %) queda en
la base «siempre hay error» (54,5 %). No es un problema de umbral ni de peso de clase (sección 11.1): el detector
no tiene con qué juzgar si una afirmación clínica es correcta. Es la falla que justifica la extensión con MedQA.

La última fila de A y B mide la localización pura: en las notas con error, ¿la oración más sospechosa es la del
error? Es la que se compara con la base «la última oración».

## 7 · Corrector BioBART (caja 6)

| Función | Qué hace |
|---|---|
| `marcar_oracion(nota, id) -> str` | Encierra la oración sospechosa entre `<err>` y `</err>` (dos tokens nuevos; `resize_token_embeddings`) |
| `armar_ejemplos_correccion(df) -> Dataset` | Entrada: la nota marcada. Objetivo: `Corrected Sentence`. Solo las notas con error |
| `entrenar_corrector(train, val) -> modelo` | `AutoModelForSeq2SeqLM` con `Seq2SeqTrainer`, entropía cruzada por token, tasa 3e-5, lote 8, hasta 5 épocas; el mejor punto según ROUGE-1 en validación |
| `generar_correccion(modelo, nota_marcada, num_beams=4) -> str` | Genera la oración corregida, con `max_new_tokens=96` |

Liberamos la GPU del detector: sus probabilidades de prueba se calculan en la sección 9 con el modelo guardado.

In [16]:
from datasets import Dataset
from rouge_score import rouge_scorer
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

detector.cpu()
torch.cuda.empty_cache()

TOK_CORRECTOR = AutoTokenizer.from_pretrained(HIPER["corrector"]["modelo"])
vocab_original = len(TOK_CORRECTOR)
TOK_CORRECTOR.add_special_tokens({"additional_special_tokens": ["<err>", "</err>"]})
print(f"Vocabulario de BioBART: {vocab_original} + 2 = {len(TOK_CORRECTOR)}")


def marcar_oracion(nota: dict[int, str], id_oracion: int) -> str:
    """Reconstruye la nota con sus oraciones oficiales y encierra la oración id_oracion entre <err> y </err>."""
    return " ".join(f"<err> {t} </err>" if i == id_oracion else t for i, t in sorted(nota.items()))


def entrada_corrector(nota: pd.Series, id_oracion: int) -> str:
    """La nota marcada y, si HIPER['corrector']['pista_medqa'] lo pide, las opciones o la respuesta de MedQA."""
    texto = marcar_oracion(nota["oraciones"], id_oracion)
    modo = HIPER["corrector"]["pista_medqa"]
    if modo == "opciones":
        texto += " || Options: " + " | ".join(candidatos_correccion(nota["medqa"]))
    elif modo == "respuesta":
        texto += " || Answer: " + nota["medqa"].respuesta
    return texto


def armar_ejemplos_correccion(df: pd.DataFrame) -> Dataset:
    """Entrada: la nota con la oración del error marcada. Objetivo: la oración corregida. Solo notas con error."""
    con_error = df[df["Error Flag"] == 1]
    return Dataset.from_dict({
        "text_id": con_error["Text ID"].tolist(),
        "entrada": [entrada_corrector(n, n["Error Sentence ID"]) for _, n in con_error.iterrows()],
        "objetivo": con_error["Corrected Sentence"].tolist(),
    })


def _tokenizar_correccion(lote: dict) -> dict:
    h = HIPER["corrector"]
    enc = TOK_CORRECTOR(lote["entrada"], truncation=True, max_length=h["max_length_entrada"])
    enc["labels"] = TOK_CORRECTOR(text_target=lote["objetivo"], truncation=True, max_length=h["max_new_tokens"])["input_ids"]
    return enc


ROUGE1 = rouge_scorer.RougeScorer(["rouge1"])


def _rouge1_validacion(prediccion) -> dict:
    preds, etiquetas = prediccion
    preds = np.where(preds < 0, TOK_CORRECTOR.pad_token_id, preds)
    etiquetas = np.where(etiquetas < 0, TOK_CORRECTOR.pad_token_id, etiquetas)
    p = TOK_CORRECTOR.batch_decode(preds, skip_special_tokens=True)
    r = TOK_CORRECTOR.batch_decode(etiquetas, skip_special_tokens=True)
    return {"rouge1": float(np.mean([ROUGE1.score(b, a)["rouge1"].fmeasure for a, b in zip(p, r)]))}


def entrenar_corrector(train: Dataset, val: Dataset):
    """Ajusta BioBART-v2-large para escribir la oración corregida a partir de la nota marcada. Evalúa ROUGE-1 en
    validación al final de cada época (generando con 4 haces) y se queda con la mejor."""
    h = HIPER["corrector"]
    fijar_semilla()
    modelo = AutoModelForSeq2SeqLM.from_pretrained(h["modelo"])
    modelo.resize_token_embeddings(len(TOK_CORRECTOR))
    # no_repeat_ngram_size=3 viene por defecto en BART y prohibiría repetir 'mEq / L' en una línea de laboratorio.
    modelo.generation_config.update(num_beams=h["num_beams"], max_new_tokens=h["max_new_tokens"],
                                    no_repeat_ngram_size=0, early_stopping=True)
    columnas = ["text_id", "entrada", "objetivo"]
    train_tok = train.map(_tokenizar_correccion, batched=True, remove_columns=columnas)
    val_tok = val.map(_tokenizar_correccion, batched=True, remove_columns=columnas)
    argumentos = Seq2SeqTrainingArguments(
        output_dir=str(MODELOS / "corrector_puntos"), learning_rate=h["tasa"],
        per_device_train_batch_size=h["lote"], per_device_eval_batch_size=h["lote"],
        num_train_epochs=h["epocas"], weight_decay=0.01, warmup_ratio=0.1, lr_scheduler_type="linear",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
        metric_for_best_model="rouge1", greater_is_better=True, predict_with_generate=True,
        generation_num_beams=h["num_beams"], bf16=DISPOSITIVO == "cuda", logging_strategy="epoch",
        report_to="none", seed=SEMILLA, save_only_model=True,
    )
    entrenador = Seq2SeqTrainer(model=modelo, args=argumentos, train_dataset=train_tok, eval_dataset=val_tok,
                                data_collator=DataCollatorForSeq2Seq(TOK_CORRECTOR, model=modelo),
                                processing_class=TOK_CORRECTOR, compute_metrics=_rouge1_validacion)
    entrenador.train()
    historial = pd.DataFrame(entrenador.state.log_history)
    modelo = entrenador.model.eval()
    modelo.historial = (historial.groupby("epoch").first()[["loss", "eval_loss", "eval_rouge1"]]
                        .rename(columns={"loss": "pérdida train", "eval_loss": "pérdida val", "eval_rouge1": "ROUGE-1 val"}))
    return modelo


@torch.no_grad()
def generar_correccion(modelo, nota_marcada: str, num_beams: int = 4, num_return_sequences: int = 1) -> str | list[str]:
    """Genera la oración corregida con beam search. Con num_return_sequences > 1 devuelve los haces en orden de
    puntaje, que es lo que usa el control de calidad."""
    enc = TOK_CORRECTOR(nota_marcada, truncation=True, max_length=HIPER["corrector"]["max_length_entrada"],
                        return_tensors="pt").to(modelo.device)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DISPOSITIVO == "cuda"):
        salida = modelo.generate(**enc, num_beams=num_beams, num_return_sequences=num_return_sequences,
                                 max_new_tokens=HIPER["corrector"]["max_new_tokens"])
    textos = [t.strip() for t in TOK_CORRECTOR.batch_decode(salida, skip_special_tokens=True)]
    return textos[0] if num_return_sequences == 1 else textos


def haces_corrector(modelo, nota: pd.Series, id_oracion: int) -> list[str]:
    """Los 4 haces del corrector para la oración id_oracion de la nota, reordenados con MedQA si HIPER lo pide."""
    haces = generar_correccion(modelo, entrada_corrector(nota, id_oracion), num_beams=HIPER["corrector"]["num_beams"],
                               num_return_sequences=HIPER["corrector"]["num_beams"])
    if HIPER["corrector"]["reordenar_medqa"]:
        haces = reordenar_con_medqa(nota, nota["oraciones"][id_oracion], haces)
    return haces


ejemplos_train, ejemplos_val = armar_ejemplos_correccion(train), armar_ejemplos_correccion(val)
largos = [len(TOK_CORRECTOR(t)["input_ids"]) for t in ejemplos_train["entrada"] + ejemplos_val["entrada"]]
print(f"Ejemplos de corrección: {len(ejemplos_train)} entrenamiento, {len(ejemplos_val)} validación")
print(f"Tokens de la nota marcada: media {np.mean(largos):.0f}, máximo {max(largos)} (límite 1.024: ninguna se corta)")
print("\nEjemplo de entrada:", ejemplos_train[0]["entrada"][-300:])
print("Objetivo:", ejemplos_train[0]["objetivo"])

Vocabulario de BioBART: 85401 + 2 = 85403


Ejemplos de corrección: 973 entrenamiento, 244 validación
Tokens de la nota marcada: media 190, máximo 457 (límite 1.024: ninguna se corta)

Ejemplo de entrada: blood pressure is 98/62 mm Hg. Examination shows increased fremitus and bronchial breath sounds over the right middle lung field. <err> After reviewing imaging, the causal pathogen was determined to be Haemophilus influenzae. </err> An x-ray of the chest showed consolidation of the right upper lobe.
Objetivo: After reviewing imaging, the causal pathogen was determined to be Streptococcus pneumoniae.


In [17]:
corrector = entrenar_corrector(ejemplos_train, ejemplos_val)
corrector.save_pretrained(MODELOS / "corrector")
TOK_CORRECTOR.save_pretrained(MODELOS / "corrector")
corrector.historial

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Map:   0%|          | 0/973 [00:00<?, ? examples/s]

Map: 100%|██████████| 973/973 [00:00<00:00, 14647.29 examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Map: 100%|██████████| 244/244 [00:00<00:00, 14465.16 examples/s]

Epoch,Training Loss,Validation Loss,Rouge1
1,2.008000,1.100575,0.245343
2,0.894600,1.059394,0.573934
3,0.593400,1.123069,0.575701
4,0.442000,1.146146,0.581177
5,0.348100,1.171864,0.586568


/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


,pérdida train,pérdida val,ROUGE-1 val
epoch,,,
1.0,2.0080,1.100575,0.245343
2.0,0.8946,1.059394,0.573934
3.0,0.5934,1.123069,0.575701
4.0,0.4420,1.146146,0.581177
5.0,0.3481,1.171864,0.586568


**Arquitectura con la forma de cada tensor.** El codificador lee la nota marcada y el decodificador devuelve, en
cada paso, una distribución sobre el vocabulario ampliado.

In [18]:
enc = TOK_CORRECTOR([ejemplos_val[0]["entrada"]], return_tensors="pt").to(corrector.device)
etq = TOK_CORRECTOR(text_target=[ejemplos_val[0]["objetivo"]], return_tensors="pt")["input_ids"].to(corrector.device)
with torch.no_grad():
    out = corrector(**enc, labels=etq)
c = corrector.config
print(f"Entrada del codificador {tuple(enc['input_ids'].shape)} → salida {tuple(out.encoder_last_hidden_state.shape)}")
print(f"Logits del decodificador (B, T, vocabulario) = {tuple(out.logits.shape)}")
print(f"d_model {c.d_model} · capas {c.encoder_layers} + {c.decoder_layers} · cabezas {c.encoder_attention_heads} × {c.d_model // c.encoder_attention_heads}")

Entrada del codificador (1, 239) → salida (1, 239, 1024)
Logits del decodificador (B, T, vocabulario) = (1, 11, 85403)
d_model 1024 · capas 12 + 12 · cabezas 16 × 64


## 8 · Control de calidad (caja 7)

| Función | Qué hace |
|---|---|
| `controlar_correccion(original, haces, umbral) -> str` | Como WangLab: si el ROUGE-L entre la oración original y la corregida baja del umbral, se prueba el siguiente de los 4 haces; si ninguno pasa, queda el primero. Si la corrección es igual a la original, se marca para revisión |
| `calibrar_umbral_calidad(val) -> float` | Elige el umbral con mayor puntaje compuesto de C en validación. No copiamos el 0,7 de WangLab |

`controlar_correccion` devuelve la corrección elegida y la marca de revisión.

La calibración usa las notas con error de validación con la oración del error marcada (la real), para medir solo
el corrector y el control de calidad. Los puntajes de los 4 haces se calculan una vez y cada umbral solo cambia
qué haz se elige.

In [19]:
from functools import cache


@cache
def _puntuador_rouge_l():
    return rouge_scorer.RougeScorer(["rougeL"])


def rouge_l(a: str, b: str) -> float:
    return _puntuador_rouge_l().score(a, b)["rougeL"].fmeasure


def controlar_correccion(original: str, haces: list[str], umbral: float) -> tuple[str, bool]:
    """Primer haz que no se aleja de la original más de lo permitido (ROUGE-L ≥ umbral); si ninguno pasa, el
    primero. Devuelve también si hay que revisarla: una corrección igual a la original no corrigió nada."""
    elegido = next((h for h in haces if rouge_l(original, h) >= umbral), haces[0])
    return elegido, _limpiar(elegido).lower() == _limpiar(original).lower()


UMBRALES_CALIDAD = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.667, 0.7, 0.8, 0.9]


def calibrar_umbral_calidad(val: pd.DataFrame) -> float:
    """Umbral de ROUGE-L con mayor puntaje compuesto de C en validación. Deja la tabla en CALIBRACION_CALIDAD."""
    global CALIBRACION_CALIDAD
    filas = []
    for u in UMBRALES_CALIDAD:
        elegidas = [controlar_correccion(o, h, u) for o, h in zip(val["original"], val["haces"])]
        filas.append({
            "umbral": u,
            "C compuesto": puntaje_compuesto([e for e, _ in elegidas], val["referencia"].tolist()),
            "cambia el 1.er haz": np.mean([e != h[0] for (e, _), h in zip(elegidas, val["haces"])]),
            "marcadas para revisión": np.mean([r for _, r in elegidas]),
        })
    CALIBRACION_CALIDAD = pd.DataFrame(filas)
    mejor = CALIBRACION_CALIDAD.loc[CALIBRACION_CALIDAD["C compuesto"].idxmax(), "umbral"]
    return float(mejor)


val_err = val[val["Error Flag"] == 1].reset_index(drop=True)
val_err["original"] = [o[i] for o, i in zip(val_err["oraciones"], val_err["Error Sentence ID"])]
val_err["referencia"] = val_err["Corrected Sentence"]
val_err["haces"] = [haces_corrector(corrector, n, n["Error Sentence ID"]) for _, n in val_err.iterrows()]
UMBRAL_CALIDAD = calibrar_umbral_calidad(val_err)
CALIBRACION_PLAN = CALIBRACION_CALIDAD.copy()  # los experimentos de 11.2 la sobrescriben
print(f"Umbral del control de calidad (mayor C en validación): {UMBRAL_CALIDAD}  ·  WangLab usaba 0,7")
CALIBRACION_CALIDAD.style.format({"C compuesto": "{:.3f}", "cambia el 1.er haz": "{:.1%}", "marcadas para revisión": "{:.1%}"})

/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BleurtSPTokenizer'. 
The class this function is called from is 'BertTokenizer'.


/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/bert_score/scorer.py:139: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:208.)
  self._baseline_vals = torch.from_numpy(


Umbral del control de calidad (mayor C en validación): 0.3  ·  WangLab usaba 0,7


,umbral,C compuesto,cambia el 1.er haz,marcadas para revisión
0,0.000000,0.601,0.0%,21.7%
1,0.100000,0.602,1.6%,21.7%
2,0.200000,0.602,1.2%,21.7%
3,0.300000,0.603,2.0%,21.7%
4,0.400000,0.601,2.0%,21.7%
5,0.500000,0.601,4.5%,22.5%
6,0.600000,0.600,7.4%,22.5%
7,0.667000,0.599,9.4%,23.0%
8,0.700000,0.599,9.4%,23.0%
9,0.800000,0.601,8.6%,25.4%


In [20]:
# ROUGE-L entre la oración con error y su corrección de referencia en validación: por qué 0,7 es demasiado alto.
rl_referencia = np.array([rouge_l(o, r) for o, r in zip(val_err["original"], val_err["referencia"])])
print(f"Mediana {np.median(rl_referencia):.3f} · con 0,7 se rechazarían {np.mean(rl_referencia < 0.7):.0%} de las correcciones correctas")
val_err.assign(primer_haz=val_err["haces"].str[0])[["original", "primer_haz", "referencia"]].head(8)

Mediana 0.667 · con 0,7 se rechazarían 57% de las correcciones correctas


,original,primer_haz,referencia
0,The patient is infected by Neisseria meningitidis.,The patient is infected by Echovirus.,The patient is infected by herpes simplex virus.
1,Cultures grew salmonella enterica.,Cultures grew Salmonella typhimurium.,Cultures grew Vibrio parahaemolyticus.
2,Laboratory studies reveal infection with JC virus.,Laboratory studies reveal infection with HSV-1.,Laboratory studies reveal infection with HSV-1.
3,The most likely causal organism is Clostridium botulinum.,The most likely causal organism is Clostridium perfringens.,The most likely causal organism is Clostridium tetani.
4,STD is discussed and a viral culture and PCR testing is ordered for Herpes simplex virus 2.,STD is discussed and a viral culture and PCR testing is ordered for Neisseria gonorrhoeae.,STD is discussed and a viral culture and PCR testing is ordered for Haemophilus ducreyi.
5,Suspected of infection with clostridium botulinum.,Suspected of infection with Staphylococcus aureus.,Suspected of infection with Streptococcus pyogenes.
6,Pathology report shows Neisseria meningitidis.,Pathology report shows Streptococcus pneumoniae.,Pathology report shows Streptococcus pneumoniae.
7,The patient is infected with tick-borne encephalitis virus.,The patient is infected with Echovirus.,The patient is infected with herpes simplex virus.


## 9 · Salida y evaluación en prueba (caja 8)

| Función | Qué hace |
|---|---|
| `formatear_salida(text_id, bandera, id, correccion) -> str` | Línea `text_id bandera id corrección` (con `-1 NA` si no hay error), el formato del *shared task* |

Ahora sí corremos todo el pipeline sobre las 574 notas de prueba, con los dos umbrales fijados en validación:
detector → decisión A y B → (si la bandera es 1) corrector → control de calidad → salida.

In [21]:
def formatear_salida(text_id: str, bandera: int, id_oracion: int, correccion: str | None) -> str:
    if bandera == 0:
        return f"{text_id} 0 -1 {NA}"
    return f"{text_id} 1 {id_oracion} {_limpiar(correccion or '')}"


def ejecutar_pipeline(df: pd.DataFrame, detector, corrector, umbral_bandera: float, umbral_calidad: float,
                      combinacion=None) -> pd.DataFrame:
    """Cajas 3 → 8 sobre notas nuevas. `combinacion` es la regresión logística de la caja 5 cuando la bandera usa
    MedQA; el reordenamiento de haces lo decide HIPER['corrector']['reordenar_medqa']."""
    detector.to(DISPOSITIVO)
    pares = armar_pares(df)
    probs = probs_por_nota(pares, _probabilidades(detector, pares))
    detector.cpu()
    torch.cuda.empty_cache()
    prob_bandera = prob_combinada(combinacion, df, probs) if combinacion is not None else None
    salida = decidir_a_y_b(df, probs, umbral_bandera, prob_bandera)
    correcciones, revisar, haces_todos = [], [], []
    for (_, nota), bandera, id_pred in zip(df.iterrows(), salida["bandera"], salida["id_pred"]):
        if bandera == 0:
            correcciones.append(None), revisar.append(False), haces_todos.append([])
            continue
        original = nota["oraciones"][id_pred]
        haces = haces_corrector(corrector, nota, id_pred)
        elegida, rev = controlar_correccion(original, haces, umbral_calidad)
        correcciones.append(elegida), revisar.append(rev), haces_todos.append(haces)
    salida["correccion"] = correcciones
    salida["revisar"] = revisar
    salida["haces"] = haces_todos
    salida["linea"] = [formatear_salida(t, b, i, c) for t, b, i, c in
                       zip(salida["Text ID"], salida["bandera"], salida["id_pred"], salida["correccion"])]
    return salida


def evaluar(df: pd.DataFrame, salida: pd.DataFrame) -> dict:
    ab = evaluar_a_y_b(salida)
    pred_c = [c if b == 1 else None for b, c in zip(salida["bandera"], salida["correccion"])]
    ref_c = df["Corrected Sentence"].tolist()
    c = {f"C · {m}": puntaje_compuesto(pred_c, ref_c, metrica=m) for m in ["rouge1", "bertscore", "bleurt", "agregado"]}
    return {**ab, **c, "marcadas para revisión": float(np.mean(salida["revisar"]))}


inicio = time.time()
salida_prueba = ejecutar_pipeline(prueba, detector, corrector, UMBRAL_BANDERA, UMBRAL_CALIDAD, combinacion=COMBINACION)
print(f"Pipeline sobre {len(prueba)} notas en {(time.time() - inicio) / 60:.1f} min")
(RESULTADOS / "predicciones_prueba.txt").write_text("\n".join(salida_prueba["linea"]) + "\n")
print("\n".join(salida_prueba["linea"].head(6)))

Pipeline sobre 574 notas en 1.2 min
ms-val-0 1 5 Culture tests indicate chlamydia trachomatis.
ms-val-1 1 9 Culture tests indicate Trichomonas vaginalis.
ms-val-2 1 24 Pathology confirms Staphylococcus aureus is the causal organism.
ms-val-3 1 4 Laboratory studies confirm that he has a parainfluenza virus infection.
ms-val-4 1 0 A previously healthy 25-year-old man comes to the physician with a suspected Bartonella henselae infection because of a 1-week history of fever and fluid release from painful lumps in his right groin.
ms-val-5 1 7 Chlamydia psittaci is the causal pathogen.


In [22]:
resultado_prueba = evaluar(prueba, salida_prueba)
m_prueba = metricas_bandera(salida_prueba["bandera"], salida_prueba["Error Flag"])
print("Matriz de confusión de A:", m_prueba["matriz"])
pd.Series(resultado_prueba).to_frame("prueba").style.format("{:.3f}")

Matriz de confusión de A: {'verdaderos_positivos': 317, 'falsos_positivos': 254, 'falsos_negativos': 2, 'verdaderos_negativos': 1}


,prueba
A · exactitud,0.554
A · precisión,0.555
A · recall,0.994
A · F1,0.712
B · exactitud (todas las notas),0.530
B · exactitud (notas con error),0.950
B · localización si se sabe que hay error,0.956
C · rouge1,0.321
C · bertscore,0.359
C · bleurt,0.306


### C con la oración real marcada

Para separar los errores del corrector de los del detector: el mismo corrector y el mismo control de calidad,
pero marcando la oración que de verdad tiene el error (solo notas con error, ROUGE-1, BERTScore, BLEURT-20 y su
promedio).

In [23]:
prueba_err = prueba[prueba["Error Flag"] == 1].reset_index(drop=True)
prueba_err["original"] = [o[i] for o, i in zip(prueba_err["oraciones"], prueba_err["Error Sentence ID"])]
prueba_err["haces"] = [haces_corrector(corrector, n, n["Error Sentence ID"]) for _, n in prueba_err.iterrows()]
corr_real = [controlar_correccion(o, h, UMBRAL_CALIDAD)[0] for o, h in zip(prueba_err["original"], prueba_err["haces"])]
c_real = puntajes_correccion(corr_real, prueba_err["Corrected Sentence"].tolist())
pd.Series({k: v for k, v in c_real.items() if k != "por_par"}).to_frame("C con la oración real").style.format("{:.3f}")

,C con la oración real
rouge1,0.596
bertscore,0.662
bleurt,0.560
agregado,0.606


### Revisión manual de las correcciones con puntaje bajo

Las métricas castigan los sinónimos, así que miramos las 10 correcciones de menor puntaje (con la oración real
marcada) antes de sacar conclusiones.

In [24]:
revision = prueba_err.assign(prediccion=corr_real, agregado=[d["agregado"] for d in c_real["por_par"]])
revision.nsmallest(10, "agregado")[["original", "prediccion", "Corrected Sentence", "agregado"]]

,original,prediccion,Corrected Sentence,agregado
266,Fetal monitoring with continuous cardiotocography is recommended.,"After four hours of observation, her vital signs are unchanged.",Labor induction is recommended.,0.071141
141,Modified radical mastectomy is required.,A core biopsy shows foam cells and multinucleated giant cells.,Reassurance is provided to the patient.,0.117688
228,Hysterectomy is required.,Planning for oxytocin administration.,Patient's posterior fornix is elevated.,0.138545
261,Liver biopsy was performed.,Hepatitis C serology was performed.,Repeat abdominal ultrasound in 6 months planned after the following results.,0.141129
157,CT scan of head is obtained.,Patient was reassured and discharged.,Brain MRI is obtained.,0.142119
173,Cesarean delivery was performed.,Plan for oxytocin administration.,Vaginal delivery was performed.,0.151628
104,Endovascular thrombolysis was performed.,Endovascular thrombolysis was performed.,Administered dalteparin.,0.152253
181,""" Dilation and curettage is recommended.",Expectant management is recommended.,""" Labor was induced.",0.159840
146,FSH and LH serum levels will be measured.,Serum FSH and LH levels will be measured.,Patient is scheduled for a hysteroscopy with potential adhesiolysis.,0.161868
152,Oral acyclovir is prescribed.,Oral azithromycin is prescribed.,Recommended curettage.,0.164925


### Tabla de referencia del reporte (§5) con nuestra fila

Nuestras cifras son sobre las 574 notas de la validación oficial de MS; las de los artículos, sobre su prueba
(MEDEC: MS + UW; WangLab: prueba de MEDIQA-CORR), así que la comparación es orientativa.

In [25]:
def fila(a, b, c):
    return {"A. Bandera": a, "B. Oración": b, "C. AggregateScore": c}


referencia = pd.DataFrame({
    "Siempre «hay error» (prueba)": fila(f"{bases(prueba)['A · siempre «hay error»']:.1%}", "—", "—"),
    "Siempre «la última oración» (prueba, notas con error)": fila("—", f"{bases(prueba)['B · última oración (notas con error)']:.1%}", "—"),
    "Claude 3.5 Sonnet, prompt (MEDEC, prueba)": fila("70,2 %", "65,6 %", "0,280"),
    "o1-preview, prompt (MEDEC, prueba)": fila("67,5 %", "61,4 %", "0,698"),
    "Médico #1 / #2 (MEDEC, prueba)": fila("79,6 % / 71,6 %", "65,9 % / 66,8 %", "0,453 / 0,712"),
    "WangLab, GPT-4 + DSPy (prueba MEDIQA-CORR)": fila("86,5 %", "83,6 %", "0,789"),
    "Nuestro pipeline (prueba)": fila(f"{resultado_prueba['A · exactitud']:.1%}",
                                      f"{resultado_prueba['B · exactitud (todas las notas)']:.1%}",
                                      f"{resultado_prueba['C · agregado']:.3f}"),
    "Nuestro pipeline, B solo notas con error (prueba)": fila("—", f"{resultado_prueba['B · localización si se sabe que hay error']:.1%}", "—"),
}).T
referencia

,A. Bandera,B. Oración,C. AggregateScore
Siempre «hay error» (prueba),55.6%,—,—
"Siempre «la última oración» (prueba, notas con error)",—,61.8%,—
"Claude 3.5 Sonnet, prompt (MEDEC, prueba)","70,2 %","65,6 %","0,280"
"o1-preview, prompt (MEDEC, prueba)","67,5 %","61,4 %","0,698"
"Médico #1 / #2 (MEDEC, prueba)","79,6 % / 71,6 %","65,9 % / 66,8 %","0,453 / 0,712"
"WangLab, GPT-4 + DSPy (prueba MEDIQA-CORR)","86,5 %","83,6 %","0,789"
Nuestro pipeline (prueba),55.4%,53.0%,0.329
"Nuestro pipeline, B solo notas con error (prueba)",—,95.6%,—


**Lectura del plan en prueba.** Se repite lo de validación. A queda en 55,4 % (la base «siempre hay error» es
55,6 %): 254 falsos positivos y solo un verdadero negativo. La localización sí funciona: 95,6 % de las notas con
error, frente al 61,8 % de la base «la última oración». C baja a 0,329 porque cada falso positivo genera una
corrección donde la referencia es NA y vale 0. Con la oración real marcada, el corrector solo llega a 0,606. La
revisión manual muestra por qué: casi todas las correcciones malas cambian un organismo o un tratamiento por otro
plausible pero incorrecto. El corrector sabe *dónde* y *qué tipo* de cosa cambiar, no cuál es la respuesta.

## 10 · Extensión · Reordenar los haces con las opciones de MedQA

`reordenar_con_medqa` (sección 2b) pone primero la sustitución de la opción incorrecta por la respuesta, si la
oración trae una, y después el primer haz que introduce una opción nueva. Después actúa el mismo control de calidad.

In [26]:
reordenar_antes = HIPER["corrector"]["reordenar_medqa"]
HIPER["corrector"]["reordenar_medqa"] = True
salida_medqa = ejecutar_pipeline(prueba, detector, corrector, UMBRAL_BANDERA, UMBRAL_CALIDAD, combinacion=COMBINACION)
resultado_medqa = evaluar(prueba, salida_medqa)
corr_real_medqa = [controlar_correccion(n["original"], reordenar_con_medqa(n, n["original"], n["haces"]), UMBRAL_CALIDAD)[0]
                   for _, n in prueba_err.iterrows()]
HIPER["corrector"]["reordenar_medqa"] = reordenar_antes
c_real_medqa = puntajes_correccion(corr_real_medqa, prueba_err["Corrected Sentence"].tolist())
pd.DataFrame({
    "pipeline": {"C compuesto (prueba)": resultado_prueba["C · agregado"], "C con la oración real": c_real["agregado"]},
    "pipeline + MedQA": {"C compuesto (prueba)": resultado_medqa["C · agregado"], "C con la oración real": c_real_medqa["agregado"]},
}).style.format("{:.3f}")

,pipeline,pipeline + MedQA
C compuesto (prueba),0.329,0.402
C con la oración real,0.606,0.741


**Lectura.** Con el detector del plan, reordenar con MedQA sube C de 0,329 a 0,402, y con la oración real de
0,606 a 0,741. A y B no cambian, porque el reordenamiento solo toca la corrección.

## 11 · Experimentos (solo validación)

Las secciones 1–10 son el plan tal cual. Aquí probamos variantes de cada caja para quedarnos con el mejor modelo,
**midiendo siempre en validación**: la prueba solo se usa una vez, en la sección 12, con la configuración que gane
aquí. Cada experimento cambia unos pocos hiperparámetros de `HIPER` y vuelve a entrenar solo lo que cambia.

**Por qué estos experimentos.** La sección 6 mostró que el detector del plan localiza el error (B) pero no sabe si
la nota lo tiene (A): su probabilidad por nota no separa las notas correctas de las que tienen error (AUC ≈ 0,5).
Probamos primero arreglos dentro del plan (sin peso de clase, un modelo más grande) y después la falla que el plan no
puede arreglar solo: el detector necesita saber cuál es la respuesta correcta. En MS esa respuesta está en la
pregunta de MedQA de la que salió la nota (sección 2b), así que la usamos como pista del detector y como rasgo de la
bandera. En la corrección probamos lo que el plan ya proponía para la extensión: las opciones de MedQA como pistas
del corrector y para reordenar sus haces.

In [27]:
from contextlib import contextmanager


@contextmanager
def con_hiper(cambios: dict):
    """Aplica cambios a HIPER dentro del bloque y deja HIPER como estaba al salir."""
    respaldo = copy.deepcopy(HIPER)
    for caja, valores in cambios.items():
        HIPER[caja].update(valores)
    try:
        yield
    finally:
        HIPER.clear()
        HIPER.update(respaldo)


def medir_bandera(df: pd.DataFrame, probs: dict[str, list[float]]) -> dict:
    """A y B en validación con la bandera del detector (el plan) y con la bandera combinada con MedQA."""
    umbral = ajustar_umbral([probs[i] for i in df["Text ID"]], df["Error Flag"].tolist())
    solo = evaluar_a_y_b(decidir_a_y_b(df, probs, umbral))
    solo["AUC por nota"] = roc_auc_score(df["Error Flag"], df["Text ID"].map(lambda t: max(probs[t])))
    _, fuera = ajustar_combinacion(df, probs)
    umbral_c = ajustar_umbral([[fuera[i]] for i in df["Text ID"]], df["Error Flag"].tolist())
    combinada = evaluar_a_y_b(decidir_a_y_b(df, probs, umbral_c, fuera))
    combinada["AUC por nota"] = roc_auc_score(df["Error Flag"], df["Text ID"].map(fuera))
    return {"bandera del detector": solo, "bandera combinada con MedQA": combinada}

### 11.1 · Detector y bandera (cajas 3–5)

| Experimento | Qué cambia | Por qué |
|---|---|---|
| plan | nada (el detector de la sección 5) | referencia |
| sin peso de clase | `pos_weight = 1` | con 18,8 el modelo puede volverse alarmista; sin peso, ¿separa mejor las notas? |
| pista MedQA | la nota va precedida de `MedQA answer: <respuesta>` | el detector puede cotejar cada oración con la respuesta correcta, aunque la nota la parafrasee |
| pista MedQA + BiomedBERT-large | el mismo, con el modelo *large* (24 capas, 1.024 dims, lote 8 × 2, tasa 1e-5) | ¿más capacidad compara mejor? |

Cada detector se mide con la bandera del plan (umbral sobre su oración más sospechosa) y con la bandera combinada:
la regresión logística de la caja 5 sobre su probabilidad y los rasgos de MedQA (sección 2b), con probabilidades
fuera de pliegue dentro de validación.

In [28]:
EXPERIMENTOS_DETECTOR = {
    "plan": {},
    "sin peso de clase": {"detector": {"pos_weight": 1.0}},
    "pista MedQA": {"detector": {"pista_medqa": True}},
    "pista MedQA + BiomedBERT-large": {"detector": {
        "pista_medqa": True, "modelo": "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract",
        "lote": 8, "acumulacion": 2, "tasa": 1e-5}},
}
if RAPIDO:
    EXPERIMENTOS_DETECTOR.pop("pista MedQA + BiomedBERT-large")

METRICAS.liberar()
corrector.cpu()
torch.cuda.empty_cache()
DETECTORES, PROBS_VAL_DETECTOR, filas_detector = {}, {}, []
for nombre, cambios in EXPERIMENTOS_DETECTOR.items():
    inicio = time.time()
    with con_hiper(cambios):
        p_train, p_val = armar_pares(train), armar_pares(val)
        modelo = detector if nombre == "plan" else entrenar_detector(p_train, p_val)
        modelo.to(DISPOSITIVO)
        probs = probs_por_nota(p_val, _probabilidades(modelo, p_val))
        medidas = medir_bandera(val, probs)
    modelo.cpu()
    torch.cuda.empty_cache()
    DETECTORES[nombre], PROBS_VAL_DETECTOR[nombre] = modelo, probs
    for bandera, m in medidas.items():
        filas_detector.append({"detector": nombre, "bandera": bandera, **m,
                               "épocas": len(modelo.historial), "minutos": (time.time() - inicio) / 60})
    print(f"{nombre}: listo en {(time.time() - inicio) / 60:.1f} min")

tabla_detector = pd.DataFrame(filas_detector).set_index(["detector", "bandera"])
columnas = ["A · exactitud", "A · F1", "AUC por nota", "B · exactitud (todas las notas)",
            "B · localización si se sabe que hay error", "épocas", "minutos"]
tabla_detector[columnas].style.format({c: "{:.1%}" for c in columnas[:5] if c != "AUC por nota"} |
                                      {"AUC por nota": "{:.3f}", "minutos": "{:.0f}"})

plan: listo en 0.2 min


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'época': 1, 'pérdida train': 0.12316698081417636, 'F1 de A (val)': 0.6974664679582713, 'AUC por nota (val)': 0.49691618450658953, 'minutos': 4.1740786790847775}


{'época': 2, 'pérdida train': 0.07546843046239063, 'F1 de A (val)': 0.6738794435857806, 'AUC por nota (val)': 0.47683622629379624, 'minutos': 4.183021116256714}


{'época': 3, 'pérdida train': 0.06528650439277933, 'F1 de A (val)': 0.6891271056661562, 'AUC por nota (val)': 0.5261672291867566, 'minutos': 4.189667693773905}
Early stopping: la época 3 no mejoró el F1 de A


sin peso de clase: listo en 12.8 min


... (8 outputs left out)

{'época': 2, 'pérdida train': 0.1348929315701593, 'F1 de A (val)': 0.968, 'AUC por nota (val)': 0.9796488267438123, 'minutos': 12.5006897966067}


{'época': 3, 'pérdida train': 0.05708848102747752, 'F1 de A (val)': 0.9759036144578312, 'AUC por nota (val)': 0.9824011571841851, 'minutos': 12.475678944587708}


{'época': 4, 'pérdida train': 0.02885599327354028, 'F1 de A (val)': 0.9696969696969696, 'AUC por nota (val)': 0.9842594824815173, 'minutos': 12.501916348934174}
Early stopping: la época 4 no mejoró el F1 de A


pista MedQA + BiomedBERT-large: listo en 50.6 min


In [29]:
# Curvas de entrenamiento: F1 de A y AUC por nota en validación, época por época.
pd.concat({n: m.historial.set_index("época")[["F1 de A (val)", "AUC por nota (val)"]] for n, m in DETECTORES.items()},
          axis=1).round(3)

plan                    sin peso de clase                     \
      F1 de A (val) AUC por nota (val)     F1 de A (val) AUC por nota (val)   
época                                                                         
1             0.707              0.527             0.697              0.497   
2             0.693              0.513             0.674              0.477   
3             0.694              0.521             0.689              0.526   
4               NaN                NaN               NaN                NaN   

        pista MedQA                    pista MedQA + BiomedBERT-large  \
      F1 de A (val) AUC por nota (val)                  F1 de A (val)   
época                                                                   
1             0.944              0.975                          0.968   
2             0.957              0.981                          0.968   
3             0.970              0.986                          0.976   
4             0.959              0.988                          0.970   

                          
      AUC por nota (val)  
época                     
1                  0.976  
2                  0.980  
3                  0.982  
4                  0.984

**Lectura de 11.1.**

* **Sin peso de clase no cambia nada** (AUC 0,50; A 54,7 %). El problema no era el `pos_weight`.
* **La pista de MedQA lo cambia todo.** Con la respuesta correcta al frente de la nota, el AUC por nota pasa de 0,53
  a 0,986 y A de 55,1 % a 96,7 %. El detector aprende a cotejar cada oración con la respuesta.
* **Los rasgos de MedQA solos ya separan las notas.** Con el detector del plan, la bandera combinada llega a 95,1 %.
  Cuando el detector ya tiene la pista, la combinación sube el AUC (0,994 con el *large*) pero no cambia A.
* **El *large* gana poco y cuesta mucho.** Sube A de 96,7 % a 97,3 % (2 notas de 448) y tarda 51 min frente a 17.
  Como el objetivo es el mejor modelo y la mejora es consistente en todas las épocas, lo dejamos.

En la comparación de abajo, la regresión logística de la caja 5 y el *boosting* quedan dentro del ruido (A entre
94 % y 97 %), así que se mantiene la logística del plan.

**¿Qué rasgos y qué modelo para la bandera combinada?** Con las probabilidades de validación del detector
«pista MedQA», comparamos la regresión logística de la caja 5 (7 rasgos) con versiones de menos rasgos y con un
*boosting* de árboles, todas fuera de pliegue dentro de validación.

In [30]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rasgos_val = tabla_rasgos(val, PROBS_VAL_DETECTOR["pista MedQA"])
conjuntos = {
    "solo el detector": ["prob_detector"],
    "solo MedQA (nota)": ["similitud", "contencion_respuesta", "contencion_otras", "diferencia"],
    "detector + MedQA (nota)": ["prob_detector", "similitud", "contencion_respuesta", "contencion_otras", "diferencia"],
    "detector + MedQA (nota y oración) · caja 5": RASGOS_BANDERA,
}
clasificadores = {"logística": lambda: make_pipeline(StandardScaler(), LogisticRegression()),
                  "boosting": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200)}
filas = []
for nombre_c, cols in conjuntos.items():
    for nombre_m, fabrica in clasificadores.items():
        X, y = rasgos_val[cols].values, val["Error Flag"].values
        fuera = np.zeros(len(y))
        for a, b in GroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA).split(X, groups=val["caso"]):
            fuera[b] = fabrica().fit(X[a], y[a]).predict_proba(X[b])[:, 1]
        barrido = barrer_umbrales(fuera, y)
        filas.append({"rasgos": nombre_c, "modelo": nombre_m, "AUC": roc_auc_score(y, fuera),
                      "A con el umbral de mayor F1": barrido.loc[barrido["f1"].idxmax(), "exactitud"]})
pd.DataFrame(filas).set_index(["rasgos", "modelo"]).style.format({"AUC": "{:.3f}", "A con el umbral de mayor F1": "{:.1%}"})

### 11.2 · Corrector y control de calidad (cajas 6–7)

Las variantes se miden como en la sección 8: notas con error de validación, con la oración del error marcada, para
aislar el corrector. Para cada una se recalibra el umbral del control de calidad.

| Experimento | Qué cambia |
|---|---|
| plan | nada (el corrector de la sección 7) |
| pista: opciones | la entrada termina en ` \|\| Options: a \| b \| c \| d` (las 4 opciones de la pregunta de MedQA) |
| pista: respuesta | la entrada termina en ` \|\| Answer: <respuesta correcta de MedQA>` |
| … + reordenar MedQA | los mismos haces, reordenados con `reordenar_con_medqa`: primero la sustitución de la opción incorrecta por la respuesta, si la oración trae una |

In [31]:
EXPERIMENTOS_CORRECTOR = {
    "plan": {},
    "pista: opciones": {"corrector": {"pista_medqa": "opciones"}},
    "pista: respuesta": {"corrector": {"pista_medqa": "respuesta"}},
}


def medir_corrector(modelo, df_err: pd.DataFrame) -> dict:
    """C en validación (oración real marcada) con el umbral de calidad calibrado, sin control y reordenando con MedQA."""
    df_err = df_err.copy()
    df_err["haces"] = [haces_corrector(modelo, n, n["Error Sentence ID"]) for _, n in df_err.iterrows()]
    umbral = calibrar_umbral_calidad(df_err)
    tabla = CALIBRACION_CALIDAD.set_index("umbral")["C compuesto"]
    reordenado = df_err.assign(haces=[reordenar_con_medqa(n, n["original"], n["haces"]) for _, n in df_err.iterrows()])
    umbral_r = calibrar_umbral_calidad(reordenado)
    tabla_r = CALIBRACION_CALIDAD.set_index("umbral")["C compuesto"]
    return {
        "sin control": {"C": tabla[0.0], "umbral": 0.0},
        "con control": {"C": tabla[umbral], "umbral": umbral},
        "+ reordenar MedQA, con control": {"C": tabla_r[umbral_r], "umbral": umbral_r},
    }


CORRECTORES, filas_corrector = {}, []
for nombre, cambios in EXPERIMENTOS_CORRECTOR.items():
    inicio = time.time()
    with con_hiper(cambios):
        if nombre == "plan":
            modelo = corrector
        else:
            METRICAS.liberar()
            modelo = entrenar_corrector(armar_ejemplos_correccion(train), armar_ejemplos_correccion(val))
        modelo.to(DISPOSITIVO)
        medidas = medir_corrector(modelo, val_err)
    modelo.cpu()
    torch.cuda.empty_cache()
    CORRECTORES[nombre] = modelo
    for variante, m in medidas.items():
        filas_corrector.append({"corrector": nombre, "control de calidad": variante, **m,
                                "ROUGE-1 val (mejor época)": modelo.historial["ROUGE-1 val"].max(),
                                "minutos": (time.time() - inicio) / 60})
    print(f"{nombre}: listo en {(time.time() - inicio) / 60:.1f} min")

tabla_corrector = pd.DataFrame(filas_corrector).set_index(["corrector", "control de calidad"])

# ¿Cuántas oraciones con error traen literal una opción incorrecta, y qué tan buena es la sustitución sola?
sustituciones = [sustituir_opcion(n["original"], n["medqa"]) for _, n in val_err.iterrows()]
aplica = [(s_, r) for s_, r in zip(sustituciones, val_err["referencia"]) if s_ is not None]
print(f"La sustitución aplica en {len(aplica)} de {len(val_err)} notas con error de validación ({len(aplica) / len(val_err):.0%});"
      f" es idéntica a la referencia en {np.mean([_limpiar(a).lower() == _limpiar(r).lower() for a, r in aplica]):.0%}"
      f" de ellas y su C es {puntajes_correccion([a for a, _ in aplica], [r for _, r in aplica])['agregado']:.3f}")
tabla_corrector.style.format({"C": "{:.3f}", "umbral": "{:.3f}", "ROUGE-1 val (mejor época)": "{:.3f}", "minutos": "{:.0f}"})

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BleurtSPTokenizer'. 
The class this function is called from is 'BertTokenizer'.


plan: listo en 0.6 min


Map:   0%|          | 0/973 [00:00<?, ? examples/s]

Map: 100%|██████████| 973/973 [00:00<00:00, 13804.05 examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Map: 100%|██████████| 244/244 [00:00<00:00, 12598.76 examples/s]

... (10 outputs left out)

/home/camilo-cachy/Personal/maestria/pln-taller-7/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BleurtSPTokenizer'. 
The class this function is called from is 'BertTokenizer'.


pista: respuesta: listo en 5.6 min
La sustitución aplica en 103 de 244 notas con error de validación (42%); es idéntica a la referencia en 65% de ellas y su C es 0.898


**Lectura de 11.2.**

* **Darle la respuesta al corrector es lo que más mejora C**: de 0,601 a 0,850. Con solo las 4 opciones llega a
  0,672, porque todavía tiene que adivinar cuál es la correcta.
* **El control de calidad casi no aporta.** Con el corrector del plan sube C de 0,601 a 0,603. Con la respuesta, el
  umbral calibrado es 0, es decir, no actúa.
* **Reordenar con MedQA ayuda cuando el corrector no sabe la respuesta** (0,601 → 0,729), pero no cuando ya la sabe
  (0,850 → 0,847). Lo que hace la sustitución, el corrector con pista ya lo aprendió.

### 11.3 · La mejor configuración

Del detector nos quedamos con la fila de mayor exactitud de A en validación (la métrica oficial; B depende de A en
la métrica oficial). Del corrector, con la de mayor C compuesto en validación.

In [32]:
mejor_det, mejor_bandera = tabla_detector["A · exactitud"].idxmax()
mejor_corr, mejor_control = tabla_corrector["C"].idxmax()
MEJOR = copy.deepcopy(EXPERIMENTOS_DETECTOR[mejor_det])
for caja, valores in EXPERIMENTOS_CORRECTOR[mejor_corr].items():
    MEJOR.setdefault(caja, {}).update(valores)
MEJOR.setdefault("decision", {})["combinar_medqa"] = mejor_bandera == "bandera combinada con MedQA"
MEJOR.setdefault("corrector", {})["reordenar_medqa"] = mejor_control.startswith("+ reordenar")
print(f"Detector: {mejor_det} · {mejor_bandera}")
print(f"Corrector: {mejor_corr} · {mejor_control}")
print("Cambios sobre el plan:", json.dumps(MEJOR, ensure_ascii=False))

Detector: pista MedQA + BiomedBERT-large · bandera del detector
Corrector: pista: respuesta · sin control
Cambios sobre el plan: {"detector": {"pista_medqa": true, "modelo": "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract", "lote": 8, "acumulacion": 2, "tasa": 1e-05}, "corrector": {"pista_medqa": "respuesta", "reordenar_medqa": false}, "decision": {"combinar_medqa": false}}


## 12 · Pipeline final en prueba

La configuración elegida en validación corre sobre las 574 notas de prueba, igual que el plan en la sección 9: los
umbrales (bandera y control de calidad) y la regresión logística de la bandera se fijan en validación.

In [33]:
detector_final, corrector_final = DETECTORES[mejor_det], CORRECTORES[mejor_corr]
with con_hiper(MEJOR):
    probs_val_final = PROBS_VAL_DETECTOR[mejor_det]
    combinacion_final = None
    if HIPER["decision"]["combinar_medqa"]:
        combinacion_final, fuera = ajustar_combinacion(val, probs_val_final)
        umbral_bandera_final = ajustar_umbral([[fuera[i]] for i in val["Text ID"]], val["Error Flag"].tolist())
    else:
        umbral_bandera_final = ajustar_umbral([probs_val_final[i] for i in val["Text ID"]], val["Error Flag"].tolist())
    umbral_calidad_final = float(tabla_corrector.loc[(mejor_corr, mejor_control), "umbral"])
    corrector_final.to(DISPOSITIVO)
    salida_final = ejecutar_pipeline(prueba, detector_final, corrector_final, umbral_bandera_final,
                                     umbral_calidad_final, combinacion=combinacion_final)
    resultado_final = evaluar(prueba, salida_final)
    haces_reales = [haces_corrector(corrector_final, n, n["Error Sentence ID"]) for _, n in prueba_err.iterrows()]
    corr_real_final = [controlar_correccion(o, h, umbral_calidad_final)[0] for o, h in zip(prueba_err["original"], haces_reales)]
    c_real_final = puntajes_correccion(corr_real_final, prueba_err["Corrected Sentence"].tolist())
m_final = metricas_bandera(salida_final["bandera"], salida_final["Error Flag"])
(RESULTADOS / "predicciones_prueba_final.txt").write_text("\n".join(salida_final["linea"]) + "\n")
print(f"Umbral de la bandera {umbral_bandera_final} · umbral de calidad {umbral_calidad_final}")
print("Matriz de confusión de A:", m_final["matriz"])
pd.DataFrame({"plan (sección 9)": resultado_prueba, "final": resultado_final}).style.format("{:.3f}")

Umbral de la bandera 0.02 · umbral de calidad 0.0
Matriz de confusión de A: {'verdaderos_positivos': 317, 'falsos_positivos': 9, 'falsos_negativos': 2, 'verdaderos_negativos': 246}


,plan (sección 9),final
A · exactitud,0.554,0.981
A · precisión,0.555,0.972
A · recall,0.994,0.994
A · F1,0.712,0.983
B · exactitud (todas las notas),0.530,0.976
B · exactitud (notas con error),0.950,0.984
B · localización si se sabe que hay error,0.956,0.991
C · rouge1,0.321,0.899
C · bertscore,0.359,0.920
C · bleurt,0.306,0.906


In [34]:
referencia_final = referencia.copy()
referencia_final.loc["Pipeline final (prueba)"] = pd.Series(fila(
    f"{resultado_final['A · exactitud']:.1%}", f"{resultado_final['B · exactitud (todas las notas)']:.1%}",
    f"{resultado_final['C · agregado']:.3f}"))
referencia_final.loc["Pipeline final, B solo notas con error (prueba)"] = pd.Series(fila(
    "—", f"{resultado_final['B · localización si se sabe que hay error']:.1%}", "—"))
referencia_final.loc["Pipeline final, C con la oración real (prueba)"] = pd.Series(fila("—", "—", f"{c_real_final['agregado']:.3f}"))
referencia_final

,A. Bandera,B. Oración,C. AggregateScore
Siempre «hay error» (prueba),55.6%,—,—
"Siempre «la última oración» (prueba, notas con error)",—,61.8%,—
"Claude 3.5 Sonnet, prompt (MEDEC, prueba)","70,2 %","65,6 %","0,280"
"o1-preview, prompt (MEDEC, prueba)","67,5 %","61,4 %","0,698"
"Médico #1 / #2 (MEDEC, prueba)","79,6 % / 71,6 %","65,9 % / 66,8 %","0,453 / 0,712"
"WangLab, GPT-4 + DSPy (prueba MEDIQA-CORR)","86,5 %","83,6 %","0,789"
Nuestro pipeline (prueba),55.4%,53.0%,0.329
"Nuestro pipeline, B solo notas con error (prueba)",—,95.6%,—
Pipeline final (prueba),98.1%,97.6%,0.908
"Pipeline final, B solo notas con error (prueba)",—,99.1%,—


**Lectura del pipeline final en prueba.**

* **A**: 98,1 % (9 falsos positivos, 2 falsos negativos).
* **B**: 97,6 % sobre todas las notas y 99,1 % de localización en las notas con error.
* **C**: 0,908 compuesto; 0,866 con la oración real marcada.

Frente al plan: +43 puntos en A, +45 en B y +0,58 en C. Supera a WangLab (GPT-4 + DSPy; 86,5 %, 83,6 %, 0,789) y a
los médicos de MEDEC, aunque no son el mismo conjunto de prueba (ver la nota de la tabla de referencia). El C final
supera al C con la oración real porque en el compuesto las notas sin error bien marcadas (NA contra NA) valen 1.

## Conclusiones

**El plan, tal como se entregó (secciones 1–10), funciona caja por caja, pero tiene una falla que no se arregla
dentro de él.**

* La limpieza (techo de B de 99,8 %), la partición por caso sin fuga, el detector, el corrector, el control de
  calidad y la evaluación oficial corren y se verifican.
* El detector localiza el error (95,6 % en prueba, contra 61,8 % de la base), pero no distingue una nota correcta de
  una con error (AUC 0,53).
* A queda en la base y arrastra a B y a C (0,329).
* La causa es de diseño, no de hiperparámetros: sin peso de clase el detector da lo mismo. En MS el error es cambiar
  la respuesta del examen por otra plausible, y nada en la nota dice cuál es la correcta.

**La extensión que el plan ya proponía (MedQA) es la que resuelve esa falla.** Recuperar la pregunta de MedQA de
cada nota (similitud mediana 0,95) y dar la respuesta correcta como pista al detector y al corrector lleva el
pipeline a 98,1 % / 97,6 % / 0,908 en prueba.

**Desviaciones del plan en la configuración final**, todas elegidas en validación (sección 11):

| Caja | Plan | Final | Motivo |
|---|---|---|---|
| 3 · Ejemplos del detector | nota tal cual | nota precedida de `MedQA answer: <respuesta>` | el detector no tiene con qué juzgar si la nota es correcta (AUC 0,53 → 0,986) |
| 4 · Detector | BiomedBERT-base, lote 16, tasa 2e-5 | BiomedBERT-large, lote 8 × 2 (acumulación), tasa 1e-5 | +0,6 puntos de A en validación; lote y tasa ajustados para que quepa en 16 GB |
| 5 · Bandera | umbral sobre el detector | igual (la bandera combinada empata en A) | — |
| 6 · Corrector | nota marcada | nota marcada + ` \|\| Answer: <respuesta>` | C de 0,601 a 0,850 en validación |
| 7 · Control de calidad | umbral calibrado | igual, el umbral calibrado da 0 | con la pista, el corrector ya no se aleja de la oración original |

Todo lo demás es el plan: los modelos de partida, la tokenización, la pérdida, las épocas, los 4 haces, la
calibración en validación, el formato de salida y las métricas oficiales.

**Limitaciones.**

1. **El resultado depende de MedQA.** El pipeline final resuelve MS porque sus notas salen de preguntas de MedQA, y
   el índice incluye la partición de validación de MedQA (sin ella, el 16,5 % de las notas no encuentra su
   pregunta). En notas clínicas reales, como las de UW, no hay pregunta que recuperar, y el pipeline volvería a
   comportarse como el del plan.
2. **La prueba es la validación oficial de MS (574 notas).** La comparación con los artículos es orientativa.
3. **Se usó un solo pliegue de la partición por caso**, sin repetición con otras semillas.

## Resumen y archivos

Guardamos las métricas, los umbrales y las predicciones en el formato del *shared task*
(`resultados/predicciones_prueba.txt` para el plan y `resultados/predicciones_prueba_final.txt` para el pipeline
final). Los modelos ajustados quedan en `modelos/` (no se versionan).

In [35]:
resumen = {
    "modo_rapido": RAPIDO,
    "particion": {"notas_train": len(train), "notas_val": len(val), "notas_prueba": len(prueba),
                  "notas_sacadas_por_fuga": int(len(fuga))},
    "limpieza": limpieza,
    "peso_clase": PESO_CLASE,
    "detector": {"historial": detector.historial.to_dict("records"), "umbral_bandera": UMBRAL_BANDERA},
    "corrector": {"historial": corrector.historial.reset_index().to_dict("records"), "umbral_calidad": UMBRAL_CALIDAD,
                  "calibracion": CALIBRACION_PLAN.to_dict("records")},
    "bases": tabla_bases.to_dict(),
    "validacion_a_y_b": resultado_val_ab,
    "validacion_diagnostico_pares": diagnostico,
    "validacion_umbral_mayor_exactitud": mejor_exactitud.to_dict(),
    "prueba": resultado_prueba,
    "prueba_matriz_A": m_prueba["matriz"],
    "prueba_C_con_oracion_real": {k: v for k, v in c_real.items() if k != "por_par"},
    "prueba_medqa": resultado_medqa,
    "prueba_medqa_C_con_oracion_real": {k: v for k, v in c_real_medqa.items() if k != "por_par"},
    "experimentos_detector": tabla_detector.reset_index().to_dict("records"),
    "experimentos_corrector": tabla_corrector.reset_index().to_dict("records"),
    "configuracion_final": MEJOR,
    "final": {"umbral_bandera": umbral_bandera_final, "umbral_calidad": umbral_calidad_final,
              "prueba": resultado_final, "prueba_matriz_A": m_final["matriz"],
              "prueba_C_con_oracion_real": {k: v for k, v in c_real_final.items() if k != "por_par"}},
}
nombre = "metricas_pipeline_rapido.json" if RAPIDO else "metricas_pipeline.json"
(RESULTADOS / nombre).write_text(json.dumps(resumen, ensure_ascii=False, indent=2, default=float))
print("Guardado:", RESULTADOS / nombre)

Guardado: /home/camilo-cachy/Personal/maestria/pln-taller-7/resultados/metricas_pipeline.json


## Probar una nota nueva

Para correr el pipeline sobre una nota que no está en el dataset (por ejemplo, una que traiga el profesor). Se pega
el texto en `NOTA` y se corre la celda. Hay que correrla **después de la sección 12**, porque usa los modelos, los
umbrales y la configuración final que quedan en memoria. La primera vez también guarda en disco el detector y el
corrector finales (`modelos/detector_final` y `modelos/corrector_final`).

* `probar_nota(NOTA)`: el pipeline final (con la pista de MedQA).
* `probar_nota(NOTA, final=False)`: el plan (secciones 1–10, sin MedQA), para comparar.

La nota puede venir como texto corrido (se parte en oraciones por `. ! ?`) o ya numerada como en el CSV
(`0 texto\n1 texto`). Si la pregunta de MedQA recuperada tiene similitud baja, la nota no salió de MedQA y el
pipeline final pierde su pista: se comporta como el plan.

In [ ]:
for nombre, modelo, tok in [("detector_final", detector_final, detector_final.tokenizador),
                             ("corrector_final", corrector_final, TOK_CORRECTOR)]:
    if not (MODELOS / nombre).exists():
        modelo.save_pretrained(MODELOS / nombre)
        tok.save_pretrained(MODELOS / nombre)
        print("Guardado:", MODELOS / nombre)


def numerar_oraciones(texto: str) -> str:
    """Texto corrido → '0 oración\n1 oración'. Si ya viene numerado como en el CSV, lo deja igual."""
    texto = texto.strip()
    if re.match(r"^\d+ ", texto):
        return texto
    partes = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", " ".join(texto.split()))
    return "\n".join(f"{i} {p}" for i, p in enumerate(partes) if p)


def probar_nota(texto: str, final: bool = True) -> pd.DataFrame:
    """Corre el pipeline completo (cajas 1 → 8) sobre una nota nueva e imprime A, B y C."""
    oraciones = numerar_oraciones(texto)
    plano = " ".join(re.sub(r"^\d+ ", "", l) for l in oraciones.split("\n"))
    df = preparar(pd.DataFrame({"Text ID": ["nota-nueva"], "Text": [plano], "Sentences": [oraciones],
                                "Error Flag": [0], "Error Sentence ID": [-1]}))
    df["medqa"] = [recuperar_pregunta(plano, k=1)[0]]
    if final:
        det, corr, u_bandera, u_calidad, comb, cambios = (detector_final, corrector_final, umbral_bandera_final,
                                                          umbral_calidad_final, combinacion_final, MEJOR)
    else:
        det, corr, u_bandera, u_calidad, comb, cambios = detector, corrector, UMBRAL_BANDERA, UMBRAL_CALIDAD, COMBINACION, {}
    with con_hiper(cambios):
        corr.to(DISPOSITIVO)
        fila = ejecutar_pipeline(df, det, corr, u_bandera, u_calidad, combinacion=comb).iloc[0]
        corr.cpu()
        torch.cuda.empty_cache()

    pregunta = df.loc[0, "medqa"]
    print("Pipeline", "final (con MedQA)" if final else "del plan (sin MedQA)")
    print(f"MedQA: pregunta recuperada con similitud {pregunta.similitud:.2f} · respuesta «{pregunta.respuesta}»")
    if pregunta.similitud < 0.5:
        print("  ⚠ similitud < 0,5: la nota no salió de MedQA y la pista no sirve")
    print(f"A · bandera: {fila['bandera']}  (probabilidad más alta {max(fila['probs']):.3f}, umbral {u_bandera})")
    print(f"B · oración: {fila['id_pred']}")
    if fila["bandera"] == 1:
        print(f"    original:   {df.loc[0, 'oraciones'][fila['id_pred']]}")
        print(f"C · corrección: {fila['correccion']}" + ("  (igual a la original: revisar)" if fila["revisar"] else ""))
        print("    los 4 haces:", *fila["haces"], sep="\n      ")
    print("Línea del shared task:", fila["linea"])
    segmentos = df.loc[0, "segmentos"]
    return pd.DataFrame({"ids oficiales": [s.ids for s in segmentos], "oración": [s.texto for s in segmentos],
                         "prob. de error": np.round(fila["probs"], 3)})


# Pegar la nota entre las comillas triples. Vacía, usa la primera nota de prueba como ejemplo.
NOTA = """

""".strip() or prueba.loc[0, "Text"]

probar_nota(NOTA)
# probar_nota(NOTA, final=False)  # el plan, para comparar
